# CMIN -- the final run for a Q1 submission

One notebook closes every experimental gap a reviewer can still point at.  Run it top to
bottom on a GPU.  It is **resumable**: after a disconnect, run it again from the top and it
skips every checkpoint and every extraction that already exists.

* It **trains ~137 runs** (about 2.5 hours on a T4) in three tiers, most important first.
* It **never deletes, moves or overwrites** anything, and never calls `record()`.
* Everything it produces lands in `protocol/final/`, and F12 zips the results into **one
  file** for you to download and send back.

| tier | what | runs |
|---|---|---|
| T1 | factor-probe arms at ten seeds, incl. the missing uniform arm; the L7@S placebo | ~47 |
| T2 | each context-conditioning point removed alone, at identical capacity | 40 |
| T3 | head-count and width sensitivity at ten seeds | 50 |

### Order of work
1. Run every cell.  Cells 1-22 are the protocol definitions, unchanged.
2. **F0** prints the queue and runs two guards: every one-point variant has exactly L6's
   parameter count, and each patched point really is constant.  Read it.
3. **F1** trains.  If the session dies, reconnect and run everything again.
4. **F2b** extracts; the analyses F3-F11 then take a few minutes.
5. Download `protocol/final/cmin_final_results.zip` and send it back.

In [ ]:
# === 1. Imports, device, config, and Colab upload ===
import os, math, time, re, itertools, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no GPU detected - training will be slow. Runtime -> Change runtime type -> GPU.')

CFG = dict(
    L=30, H=30, HORIZONS=[1,3,7,14,30],
    d=64, d_ctx=32,
    lr=1e-3, weight_decay=1e-4, grad_clip=5.0,
    epochs=15, patience=5, batch=32, n_seeds=3,
    sigma_quantile=0.25, knn=8,
    regime_horizon=7,
    context_alpha=True,
    lambda_context_entropy=1e-3,
    lambda_alpha_phys=5e-3,
    edge_threshold=0.02,
    baseline_epochs=8, baseline_patience=3, run_baselines=True,
    bootstrap_B=1000,
    policy_half_life_days=90,
)

# --- scenario-suite knobs -------------------------------------------------------------
SCFG = dict(
    seeds=10,             # every scenario
    epochs=50,
    patience=10,
    spike_threshold=0.10, # |log move| defining a price spike
    news_half_life=7,     # default decay; S2 sweeps this
    news_share_norm=True, # divide national counts by daily indexed volume
    horizon_weight_clip=5.0,
    lambda_spike=0.3,     # weight of the S6 auxiliary BCE term
    bootstrap_B=1000,
    lambda_rest=1.0,      # weight on the REST-TS residual-supervision term
)


# ---------------------------------------------------------------------------------
# Google Drive is mounted HERE (not later) so the input files can be resolved from it.
# Upload the 10 inputs to MyDrive/cmin_data ONCE; every future session — and any other
# Google account — picks them up automatically. Checkpoints and outputs also live on
# Drive, so a disconnect costs one run rather than the suite.
# ---------------------------------------------------------------------------------
import shutil

NEEDED = [
    'dataset_train.xlsx', 'dataset_test.xlsx', 'Matrix_Jarak_Jawa_Timur_simetris.xlsx',
    'nasa_power_jawa_timur_merged.csv', '06_ConsumerPriceIndicies.xlsx',
    'policy_event_features.csv', 'policy_extraction_review_queue.csv',
    'policy_feature_dictionary.csv',
    'news_channels_national.csv.gz', 'news_channels_city.csv.gz',
]

DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
except Exception:
    print('not on Colab (or Drive refused) - using local paths')

DATA_DIR = f'{DRIVE}/cmin_data'   if DRIVE else '.'
CKPT_DIR = f'{DRIVE}/cmin_v4_ckpt' if DRIVE else 'cmin_v4_ckpt'
OUT_DIR  = f'{DRIVE}/cmin_v4_out'  if DRIVE else '.'
for d in ([DATA_DIR, CKPT_DIR, OUT_DIR] if DRIVE else [CKPT_DIR]):
    os.makedirs(d, exist_ok=True)

# 1) pull whatever Drive already has into the working directory
if DRIVE:
    for f in NEEDED:
        src = os.path.join(DATA_DIR, f)
        if not os.path.exists(f) and os.path.exists(src):
            shutil.copy(src, f)

# 2) ask for anything still missing, then persist it to Drive for next time
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'{len(missing)} file(s) not found in {DATA_DIR}:')
    for f in missing: print('   -', f)
    try:
        from google.colab import files
        print('\nUpload them now (they will be saved to Drive automatically):')
        files.upload()
    except Exception:
        raise FileNotFoundError(f'Place these next to the notebook: {missing}')
    if DRIVE:
        for f in NEEDED:
            if os.path.exists(f) and not os.path.exists(os.path.join(DATA_DIR, f)):
                shutil.copy(f, os.path.join(DATA_DIR, f))
                print(f'  saved to Drive: {f}')

missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f'still missing: {missing}'
print(f'\nall {len(NEEDED)} inputs ready')
print(f'  data       {DATA_DIR}')
print(f'  checkpoints{"":1s} {CKPT_DIR}')
print(f'  outputs    {OUT_DIR}')

In [ ]:
# === 2. Load price data; join NASA/POWER weather; load Food CPI and policy events ===
tr = pd.read_excel('dataset_train.xlsx')
te = pd.read_excel('dataset_test.xlsx')
for d in (tr, te):
    d['tanggal'] = pd.to_datetime(d['tanggal'])
df = pd.concat([tr, te], ignore_index=True).sort_values('tanggal').copy()

# NASA/POWER is daily and location-specific. Prefixing avoids silently reusing weather
# columns that happened to be present in the original price workbook.
nasa = pd.read_csv('nasa_power_jawa_timur_merged.csv', parse_dates=['date'])
# Every meteorological column the product ships with, kept for reporting in E2.
NASA_META_COLS = ['wilayah', 'date', 'latitude', 'longitude', 'elevation_m', 'source_file', 'YEAR', 'DOY']
NASA_ALL_VARS = [c for c in nasa.columns if c not in NASA_META_COLS]
NASA_DATE_MIN, NASA_DATE_MAX, NASA_ROWS = nasa['date'].min(), nasa['date'].max(), len(nasa)
NASA_FEATURES = ['PRECTOTCORR', 'GWETTOP', 'PS', 'QV2M', 'TS', 'T2M', 'T2M_MIN', 'T2M_MAX', 'RH2M', 'WS2M']
assert set(NASA_FEATURES).issubset(nasa.columns)
nasa = nasa.rename(columns={'wilayah': 'kab_kota', 'date': 'tanggal'})
nasa = nasa[['kab_kota', 'tanggal', *NASA_FEATURES]].copy()
nasa = nasa.rename(columns={c: f'nasa_{c}' for c in NASA_FEATURES})
assert not nasa.duplicated(['kab_kota', 'tanggal']).any(), 'NASA key must be unique'
df = df.merge(nasa, on=['kab_kota', 'tanggal'], how='left', validate='many_to_one')
nasa_cols = [f'nasa_{c}' for c in NASA_FEATURES]
missing_weather = int(df[nasa_cols].isna().any(axis=1).sum())
assert missing_weather == 0, f'{missing_weather} price rows lack NASA weather after the merge'

MONTH_NUM = {m: i for i, m in enumerate([
    'january','february','march','april','may','june',
    'july','august','september','october','november','december'], start=1)}

def load_food_cpi(path):
    """Parse BI's presentation-format CPI workbook and return its Foods component.

    Base-year changes make raw CPI levels incomparable across regimes, so the model uses
    only MoM/YoY percentage changes when both observations have the same CPI base.
    """
    raw = pd.read_excel(path, sheet_name=0, header=None)
    label_row = raw.iloc[5].fillna('').astype(str).str.strip().str.lower()
    food_col = label_row[label_row.eq('foods')].index[0]
    period_col = 3  # 'End of Period' field in the BI sheet
    rows, year, base_id = [], None, None
    for _, row in raw.iterrows():
        label = str(row.iloc[period_col]).strip()
        year_match = re.search(r'\b(19|20)\d{2}\b', label)
        if year_match:
            year = int(year_match.group())
            base_match = re.search(r'(\d{4}\s*=\s*100)', label)
            base_id = base_match.group(1).replace(' ', '') if base_match else label
        month = MONTH_NUM.get(label.lower())
        value = pd.to_numeric(row.iloc[food_col], errors='coerce')
        if year and month and pd.notna(value):
            rows.append((pd.Timestamp(year, month, 1), float(value), base_id))
    cpi = pd.DataFrame(rows, columns=['month', 'food_cpi_level', 'cpi_base']).drop_duplicates('month', keep='last').sort_values('month')
    cpi['food_cpi_mom_pct'] = 100 * cpi['food_cpi_level'].pct_change()
    cpi.loc[cpi['cpi_base'].ne(cpi['cpi_base'].shift()), 'food_cpi_mom_pct'] = np.nan
    cpi['food_cpi_yoy_pct'] = 100 * (cpi['food_cpi_level'] / cpi['food_cpi_level'].shift(12) - 1)
    cpi.loc[cpi['cpi_base'].ne(cpi['cpi_base'].shift(12)), 'food_cpi_yoy_pct'] = np.nan
    # The value released for a calendar month is treated as observable from the next
    # month. This is intentionally conservative for daily forecasts.
    cpi['month_available'] = cpi['month'] + pd.offsets.MonthBegin(1)
    return cpi[['month_available', 'food_cpi_mom_pct', 'food_cpi_yoy_pct']]

food_cpi = load_food_cpi('06_ConsumerPriceIndicies.xlsx')
df['cpi_month'] = df['tanggal'].dt.to_period('M').dt.to_timestamp()
df = df.merge(food_cpi, left_on='cpi_month', right_on='month_available', how='left', validate='many_to_one').drop(columns=['month_available'])

# Audit CSVs are loaded for provenance; only policy_event_features.csv supplies model inputs.
policy_dictionary = pd.read_csv('policy_feature_dictionary.csv')
policy_review_queue = pd.read_csv('policy_extraction_review_queue.csv')
policy_events_all = pd.read_csv('policy_event_features.csv', parse_dates=['event_date'])
policy_events = policy_events_all.query('include_in_model == 1').copy()
assert not policy_events['event_date'].isna().any()
POLICY_EVENT_COLS = [
    'expected_price_pressure', 'price_relevance',
    'rice', 'chili', 'shallot', 'garlic', 'beef', 'buffalo_meat', 'chicken_meat',
    'chicken_eggs', 'sugar', 'cooking_oil', 'soybean', 'corn', 'wheat', 'fish',
    'price_management', 'supply_reserve', 'procurement_distribution', 'import_export',
    'market_operation', 'production', 'subsidy_assistance',
]
assert set(POLICY_EVENT_COLS).issubset(policy_events.columns)

def policy_decay_features(event_table, daily_dates, cols, half_life_days=90):
    """National policy features; events begin on the following day to avoid look-ahead."""
    idx = pd.DatetimeIndex(daily_dates)
    ev = event_table.groupby('event_date')[cols].sum().reindex(idx, fill_value=0.0)
    ev = ev.shift(1, fill_value=0.0)  # no same-day policy information in the forecast input
    decay = 0.5 ** (1.0 / half_life_days)
    out = np.zeros((len(idx), len(cols)), dtype=np.float32)
    state = np.zeros(len(cols), dtype=np.float32)
    for t, values in enumerate(ev.to_numpy(np.float32)):
        state = decay * state + values
        out[t] = state
    return pd.DataFrame(out, index=idx, columns=[f'policy_{c}_90d' for c in cols])

daily_policy = policy_decay_features(policy_events, pd.DatetimeIndex(sorted(df['tanggal'].unique())), POLICY_EVENT_COLS, CFG['policy_half_life_days'])

# a node is a (city, market) pair, because market names repeat across cities
df['node'] = df['kab_kota'].astype(str) + ' | ' + df['pasar'].astype(str)
nodes = sorted(df['node'].unique())
cities = sorted(df['kab_kota'].unique())
N, R = len(nodes), len(cities)
dates = pd.DatetimeIndex(np.sort(df['tanggal'].unique()))
T = len(dates)
TRAIN_MAX = pd.Timestamp(tr['tanggal'].max())
TEST_START = pd.Timestamp(te['tanggal'].min())
print(f'N(nodes)={N}  R(cities)={R}  T(days)={T}  train<= {TRAIN_MAX.date()}  test>= {TEST_START.date()}')
print(f'NASA variables available: {len(NASA_ALL_VARS)}; joined: {len(NASA_FEATURES)}; rows: {NASA_ROWS:,} '
      f'({NASA_DATE_MIN.date()} to {NASA_DATE_MAX.date()})')
print(f'Food CPI range: {food_cpi.month_available.min().date()} to {food_cpi.month_available.max().date()}')
print(f'policy documents: {len(policy_events_all)} retrieved, {len(policy_events)} used; '
      f'review-queue records: {len(policy_review_queue)}')

# assignment matrix S (node -> city) and its column-normalised pooling form
cidx = {c:i for i,c in enumerate(cities)}
node_city = df.drop_duplicates('node').set_index('node')['kab_kota'].to_dict()
S = np.zeros((N, R), np.float32)
for j,n in enumerate(nodes):
    S[j, cidx[node_city[n]]] = 1.0
Sbar = S / S.sum(0, keepdims=True)

# physical graph from the distance matrix (local Gaussian kernel + kNN sparsify)
def _norm(s):
    return (str(s).replace('Kabupaten ','').replace('Kota ','')
            .replace(' Kab.','').replace('Kab. ','').strip().lower())
Dm = pd.read_excel('Matrix_Jarak_Jawa_Timur_simetris.xlsx')
dist_cities = list(Dm.iloc[:,0]); M = Dm.iloc[:,1:].to_numpy(float)
sigma = np.quantile(M[M>0], CFG['sigma_quantile'])
A = np.exp(-(M**2)/sigma**2); np.fill_diagonal(A, 0.0)
if CFG['knn']:
    keep = np.zeros_like(A)
    for a in range(R):
        ix = np.argsort(A[a])[-CFG['knn']:]; keep[a, ix] = A[a, ix]
    A = keep
A[A < 1e-3] = 0.0
rs = A.sum(1, keepdims=True); rs[rs==0] = 1.0; A = A / rs
perm = np.array([{_norm(c):i for i,c in enumerate(dist_cities)}[_norm(c)] for c in cities])
A_phys = A[np.ix_(perm, perm)].astype(np.float32)
print('A_phys', A_phys.shape, 'density', round(float((A_phys>0).mean()),3))

In [ ]:
# === H0. Repair the physical graph, BEFORE anything reads it ===
# This must run between cell 2 (which builds A_phys) and cell 3, because every downstream
# cell consumes A_phys.
#
# The defect: _norm() in cell 2 strips both "Kabupaten " and "Kota ", so the distance
# workbook's rows "Blitar Kab." and "Kota Blitar" collapse to one dictionary key. The lookup
# keeps the last, every Kabupaten inherits its Kota twin's distances, and seven genuine rows
# are discarded. Because row normalisation runs BEFORE the reindex, the reindexed rows then
# no longer sum to one (mean 0.9264, min 0.7305).
#
# Why this is now load-bearing rather than cosmetic. The paper's argument rests on the
# physical anchor contributing nothing and on distance not mattering. With a broken distance
# matrix a referee can reply that the anchor did nothing BECAUSE the matrix was wrong -- which
# would be fatal to the central claim. The claim is only worth making on a correct A_phys.
def _norm_kk(s):
    """Map a dataset city name onto its workbook row, preserving Kabupaten vs Kota.

    The workbook labels regencies that have a same-named city as 'X Kab.' and the city as
    'Kota X'; regencies with no city twin are bare 'X'. Try the specific form first.
    """
    s = str(s).strip()
    low = s.lower()
    if low.startswith('kabupaten '):
        base = s[10:].strip()
        for cand in (base + ' Kab.', base):
            if cand.lower() in _WB:
                return _WB[cand.lower()]
    elif low.startswith('kota '):
        if low in _WB:
            return _WB[low]
    return _WB.get(low)


_WB = {str(c).strip().lower(): i for i, c in enumerate(dist_cities)}
_perm_fixed = [_norm_kk(c) for c in cities]
assert all(p is not None for p in _perm_fixed), \
    'unmatched cities: %s' % [c for c, p in zip(cities, _perm_fixed) if p is None]
perm = np.array(_perm_fixed)
assert len(set(perm.tolist())) == R, 'still collapsing rows: %d unique for %d cities' % (
    len(set(perm.tolist())), R)

# Reindex first, normalise second -- the order the original had backwards.
A_phys = A[np.ix_(perm, perm)].astype(np.float32)
np.fill_diagonal(A_phys, 0.0)
A_phys = (A_phys / np.maximum(A_phys.sum(1, keepdims=True), 1e-8)).astype(np.float32)

print('A_phys repaired')
print('  unique workbook rows used : %d of %d cities (was 31)' % (len(set(perm.tolist())), R))
print('  row sums                  : mean %.4f  min %.4f (was 0.9264 / 0.7305)'
      % (A_phys.sum(1).mean(), A_phys.sum(1).min()))
print('  density                   : %.3f' % float((A_phys > 0).mean()))
print('  Kabupaten/Kota pairs now distinct, e.g. Malang:',
      perm[cities.index('Kabupaten Malang')], 'vs', perm[cities.index('Kota Malang')])
print('')
print('  Checkpoints written under the OLD A_phys are now stale. Delete the protocol')
print('  directory before the long run, or every resumed rung silently mixes two graphs.')

In [ ]:
# === 3. Leakage-safe feature engineering -> full arrays ===
def panel(col):
    return df.pivot_table(index='tanggal', columns='node', values=col,
                          aggfunc='first').reindex(index=dates, columns=nodes)

def fill_time(P):
    return pd.DataFrame(P).ffill().bfill().to_numpy()

def z_from_array(P):
    P = np.asarray(P, float)
    m = np.nanmean(P[istrain]); s = np.nanstd(P[istrain])
    if not np.isfinite(s) or s < 1e-6: s = 1.0
    return np.nan_to_num(fill_time((P - m) / s)).astype(np.float32)

def zfeat(col):
    return z_from_array(panel(col).to_numpy(float))

def anomaly_feat(col):
    P = panel(col)
    train = P.iloc[np.where(istrain)[0]]
    clim = train.groupby(train.index.dayofyear).mean()
    clim = clim.reindex(np.arange(1, 367)).interpolate(limit_direction='both').ffill().bfill()
    baseline = clim.reindex(dates.dayofyear).to_numpy(float)
    return z_from_array(P.to_numpy(float) - baseline)

price = panel('harga').to_numpy(float)
istrain = np.asarray(dates <= TRAIN_MAX)

def winsor(x, lo=1, hi=99):
    out = x.copy()
    for j in range(x.shape[1]):
        col = x[istrain, j]; col = col[~np.isnan(col)]
        if len(col) > 10:
            a, b = np.percentile(col, [lo, hi]); out[:, j] = np.clip(out[:, j], a, b)
    return out

Lp = np.log(winsor(price)); obs = (~np.isnan(price)).astype(np.float32)
mu = np.nanmean(np.where(istrain[:,None], Lp, np.nan), 0)
sd = np.nanstd(np.where(istrain[:,None], Lp, np.nan), 0); sd[sd < 1e-6] = 1.0
Z = (Lp - mu) / sd
Zf = fill_time(Z).astype(np.float32); Y = np.nan_to_num(Z).astype(np.float32)

# NASA/POWER weather is the authoritative weather source for this augmented model.
weather_cols = ['nasa_PRECTOTCORR','nasa_GWETTOP','nasa_PS','nasa_QV2M','nasa_TS','nasa_T2M','nasa_RH2M','nasa_WS2M']
weather = [zfeat(c) for c in weather_cols]
weather_anom = [anomaly_feat(c) for c in weather_cols]
rain = weather[0]; rain_anom = weather_anom[0]
harvest_raw = panel('Panen tahun lalu').to_numpy(float)
harvest = z_from_array(harvest_raw)
harvest_pressure = np.maximum(harvest, 0).astype(np.float32)
rain_harvest = z_from_array(rain_anom * harvest_pressure)
harvest_hi = np.nanquantile(harvest_raw[istrain], 0.75)
production_season = np.nan_to_num((harvest_raw >= harvest_hi).astype(np.float32))
agro = [harvest, harvest_pressure, rain_harvest, production_season]

rain_city_mean = np.nanmean(panel('nasa_PRECTOTCORR').to_numpy(float), axis=1)
harvest_city_mean = np.nanmean(harvest_raw, axis=1)
peak_rain_flag = (rain_city_mean >= np.nanquantile(rain_city_mean[istrain], 0.75)).astype(int)
harvest_flag = (harvest_city_mean >= np.nanquantile(harvest_city_mean[istrain], 0.75)).astype(int)

city_harvest_train, city_market_count = [], []
for c in cities:
    ix = np.where(S[:, cidx[c]] > 0)[0]
    city_harvest_train.append(np.nanmean(harvest_raw[istrain][:, ix])); city_market_count.append(len(ix))
city_harvest_train = np.asarray(city_harvest_train, float); city_market_count = np.asarray(city_market_count, float)
if np.isnan(city_harvest_train).any(): city_harvest_train = np.nan_to_num(city_harvest_train, nan=np.nanmedian(city_harvest_train))
production_city_mask = city_harvest_train >= np.nanquantile(city_harvest_train, 0.75)
prod_s = np.nanstd(city_harvest_train); prod_s = prod_s if np.isfinite(prod_s) and prod_s > 1e-6 else 1.0
market_s = city_market_count.std(); market_s = market_s if np.isfinite(market_s) and market_s > 1e-6 else 1.0
prod_z = (city_harvest_train - np.nanmean(city_harvest_train)) / prod_s
market_z = (city_market_count - city_market_count.mean()) / market_s
consumption_proxy_score = market_z - prod_z
consumption_city_mask = consumption_proxy_score >= np.nanquantile(consumption_proxy_score, 0.75)

dow = dates.dayofweek.to_numpy(); doy = dates.dayofyear.to_numpy()
cal = np.stack([np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7), np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)], 1).astype(np.float32)
cal_node = np.repeat(cal[:,None,:], N, 1)

NODE_FEATURE_SLICES = {
    'price': list(range(0, 2)),
    'weather': list(range(2, 2 + 2*len(weather_cols))),
    'supply': list(range(2 + 2*len(weather_cols), 2 + 2*len(weather_cols) + len(agro))),
    'calendar': list(range(2 + 2*len(weather_cols) + len(agro), 2 + 2*len(weather_cols) + len(agro) + 4)),
}
DOMAIN_NAMES = ['price', 'weather', 'supply', 'economy', 'policy']
Xnode = np.concatenate([Zf[...,None], obs[...,None], np.stack(weather,-1), np.stack(weather_anom,-1), np.stack(agro,-1), cal_node], -1).astype(np.float32)
Fn = Xnode.shape[-1]

def z_global(P):
    P = np.asarray(P, float)
    m = np.nanmean(P[istrain], axis=0); s = np.nanstd(P[istrain], axis=0)
    s = np.where(np.isfinite(s) & (s >= 1e-6), s, 1.0)
    return np.nan_to_num((P - m) / s).astype(np.float32)

# Economy: fuel, exchange rate, and leakage-safe BI Food CPI. Ramadan remains a
# known calendar signal. Policy is kept in a separate global domain and never has
# future values supplied to the model.
ECON_NUM_COLS = ['Harga BBM (Pertalite)', 'Harga BBM (Pertamax)', 'Harga BBM (Solar Subsidi)', 'Nilai Kurs', 'food_cpi_mom_pct', 'food_cpi_yoy_pct']
gdf = df.groupby('tanggal')[ECON_NUM_COLS + ['Ramadan']].first().reindex(dates)
econ_num = z_global(gdf[ECON_NUM_COLS].to_numpy(float))
ramadan_flag = gdf['Ramadan'].fillna(0).to_numpy(np.float32).astype(int)
future_ramadan = np.zeros((T, CFG['H']), np.float32)
for h in range(1, CFG['H']+1):
    src = np.arange(T) + h; ok = src < T
    future_ramadan[ok, h-1] = ramadan_flag[src[ok]]
RAMADAN_IDX = econ_num.shape[1]
FUTURE_RAMADAN_SLICE = slice(RAMADAN_IDX + 1, RAMADAN_IDX + 1 + CFG['H'])
G_economy = np.concatenate([econ_num, ramadan_flag[:,None], future_ramadan], axis=1).astype(np.float32)

policy_frame = daily_policy.reindex(dates).fillna(0.0)
G_policy = z_global(policy_frame.to_numpy(float))
GLOBAL_FEATURE_SLICES = {
    'economy': list(range(G_economy.shape[1])),
    'policy': list(range(G_economy.shape[1], G_economy.shape[1] + G_policy.shape[1])),
}
Xglobal = np.concatenate([G_economy, G_policy], axis=1).astype(np.float32)
Fg = Xglobal.shape[1]

mu_t = torch.tensor(mu, dtype=torch.float32); sd_t = torch.tensor(sd, dtype=torch.float32)
print('Xnode', Xnode.shape, '| Fn=', Fn, '| Xglobal', Xglobal.shape, '| Fg=', Fg)
print('domains:', DOMAIN_NAMES, '| policy feature count:', G_policy.shape[1])
print('known-future Ramadan columns:', future_ramadan.shape[1], '| policy has no future columns')
print('production-city proxy:', [cities[i] for i in np.where(production_city_mask)[0]])
print('consumption-city proxy:', [cities[i] for i in np.where(consumption_city_mask)[0]])

In [ ]:
# === 3b. Freeze the baseline arrays; build spike targets ===
# Everything above this line is the production pipeline, untouched. Scenarios are
# constructed by *adding to* these frozen arrays, so the baseline can never drift.
XNODE_BASE   = Xnode.copy()
XGLOBAL_BASE = Xglobal.copy()
NODE_SLICES_BASE   = {k: list(v) for k, v in NODE_FEATURE_SLICES.items()}
GLOBAL_SLICES_BASE = {k: list(v) for k, v in GLOBAL_FEATURE_SLICES.items()}
DOMAINS_BASE = list(DOMAIN_NAMES)
PRICE_IDX = 0                      # Xnode[...,0] is the normalised log price (Zf)

# --- policy expert -------------------------------------------------------------------
# 'policy' is NOT a news signal. It comes from policy_event_features.csv: 91 JDIH legal
# documents (22 with include_in_model=1), each carrying document_title, document_type,
# legal_number and legal_year - a separate regulation-PDF pipeline.
#
# It is removed from the DEFAULT baseline so that news is the only document-derived
# signal, and its contribution cannot be confounded with a second textual source. The
# block is kept in memory because the S8 scenarios fuse it back in with news.
G_POLICY_BASE = G_policy.copy()
POLICY_COLS = list(POLICY_EVENT_COLS)

DROP_POLICY = True
if DROP_POLICY and 'policy' in GLOBAL_SLICES_BASE:
    _keep = list(GLOBAL_SLICES_BASE['economy'])
    XGLOBAL_BASE = XGLOBAL_BASE[:, _keep].copy()
    GLOBAL_SLICES_BASE = {'economy': list(range(len(_keep)))}
    DOMAINS_BASE = [d for d in DOMAINS_BASE if d != 'policy']
    print(f'policy expert dropped from baseline: Fg {Xglobal.shape[1]} -> {XGLOBAL_BASE.shape[1]}')

# Why fusion is worth a scenario: the regulation record is extremely sparse next to the
# news corpus, and sparse in exactly the place this paper cares about.
_pu = policy_events
_nzp = _pu[POLICY_COLS].astype(float).sum()
print(f'policy events used: {len(_pu)}  span {_pu.event_date.min().date()} .. {_pu.event_date.max().date()}')
print(f'  -> no policy signal at all before {_pu.event_date.min().date()}, '
      f'i.e. the first {int((_pu.event_date.min() - dates[0]).days)} days of the panel')
print(f'  -> chili-tagged policy events: {int(_nzp["chili"])} of {len(_pu)}')
print(f'  -> all-zero policy columns: {[c for c in POLICY_COLS if _nzp[c] == 0]}')

print('baseline frozen: Fn =', XNODE_BASE.shape[-1], ' Fg =', XGLOBAL_BASE.shape[-1],
      ' domains =', DOMAINS_BASE)

# Spike target: |log price(t+h) - log price(t)| > threshold, in Rupiah terms.
# Y is per-node z-scored log price, so the log move is |dY| * sd[node].
THR = float(SCFG['spike_threshold'])
SPIKE = np.zeros((T, N, CFG['H']), np.float32)
for h in range(1, CFG['H'] + 1):
    dlog = np.abs(Y[h:] - Y[:-h]) * sd[None, :]
    SPIKE[:-h, :, h - 1] = (dlog > THR).astype(np.float32)
print('spike rate (|move| > %.0f%%) by horizon:' % (100 * THR),
      {h: round(float(SPIKE[:, :, h - 1].mean()), 4) for h in CFG['HORIZONS']})

In [ ]:
# === 3c. News channels from news.sqlite ===
# The database itself is not shipped to Colab. build_news_channels.py aggregated it to
# daily channels under strict look-ahead rules (available_date = pub_date + 1 unless the
# article carries an exact timestamp before 06:00 WIB; missing days are zero, never
# back-filled). Body-level annotation throughout - headline and body tagging have
# different coverage and must not be mixed.
NEWS_CHANNELS = ['supply_shock', 'logistics', 'trade_policy', 'production',
                 'supply_reserve', 'chili_focus', 'price_report', 'pressure_net']
NEWS_STRATA = ['all', 'fwd', 'price']

_nat = pd.read_csv('news_channels_national.csv.gz', parse_dates=['available_date'])
_cty = pd.read_csv('news_channels_city.csv.gz', parse_dates=['available_date']).copy()

# Decay is applied over the FULL news history and only then sliced to the price panel.
# Clipping first would start March 2021 from an empty memory and silently discard the
# January-February news that a 30-day half-life should still be carrying.
NEWS_DATES = pd.date_range(min(_nat['available_date'].min(), dates[0]),
                           max(_nat['available_date'].max(), dates[-1]), freq='D')
DATE_POS = NEWS_DATES.get_indexer(dates)
assert (DATE_POS >= 0).all(), 'price-panel dates must lie inside the news date range'

_nat = _nat.set_index('available_date').reindex(NEWS_DATES).fillna(0.0)
_vol = _nat['n_indexed'].to_numpy(float)

_cty = _cty[_cty['city'].isin(cities) & _cty['available_date'].isin(NEWS_DATES)].copy()
_cty['_ci'] = _cty['city'].map(cidx)
_ti = pd.Series(np.arange(len(NEWS_DATES)), index=NEWS_DATES).reindex(
    _cty['available_date']).to_numpy()
_ci = _cty['_ci'].to_numpy()

NEWS_RAW_NAT, NEWS_RAW_CITY = {}, {}
for s in NEWS_STRATA:
    cols = [s + '__' + c for c in NEWS_CHANNELS]
    NEWS_RAW_NAT[s] = _nat[cols].to_numpy(np.float32)                  # (Tn, K)
    arr = np.zeros((len(NEWS_DATES), R, len(NEWS_CHANNELS)), np.float32)
    arr[_ti, _ci, :] = _cty[cols].to_numpy(np.float32)
    NEWS_RAW_CITY[s] = arr                                             # (Tn, R, K)

CITY_OF_NODE = np.array([cidx[node_city[n]] for n in nodes])

def news_decay(x, half_life):
    """Exponential memory. A news shock does not stop mattering the day after it prints."""
    if not half_life or half_life <= 0:
        return x.astype(np.float32)
    lam = 0.5 ** (1.0 / float(half_life))
    out = np.zeros_like(x, dtype=np.float32)
    st = np.zeros(x.shape[1:], np.float32)
    for t in range(x.shape[0]):
        st = lam * st + x[t]
        out[t] = st
    return out

def z_train_only(x):
    """Normalise with TRAIN statistics only. Using full-series statistics would leak the
    test-period distribution into training - the single most common silent error here."""
    m = x[istrain].mean(0); s = x[istrain].std(0)
    s = np.where(np.isfinite(s) & (s > 1e-6), s, 1.0)
    return np.nan_to_num((x - m) / s).astype(np.float32)

def build_news(stratum='all', half_life=None, share_norm=None):
    """-> (national (T,K), per-node (T,N,K)), decayed over the full news history,
    sliced to the price panel, then normalised on train statistics only."""
    hl = SCFG['news_half_life'] if half_life is None else half_life
    sn = SCFG['news_share_norm'] if share_norm is None else share_norm
    nat = NEWS_RAW_NAT[stratum].copy()
    if sn:
        # Indexed volume drifts heavily across 2021-2026; raw counts would encode that
        # platform drift as if it were a change in the news itself.
        nat = nat / np.maximum(_vol, 1.0)[:, None]
    nat = z_train_only(news_decay(nat, hl)[DATE_POS])
    cty = z_train_only(news_decay(NEWS_RAW_CITY[stratum], hl)[DATE_POS])
    return nat, cty[:, CITY_OF_NODE, :]

_n, _c = build_news('all')
print('news channels:', NEWS_CHANNELS)
print('national', _n.shape, ' per-node', _c.shape)
print('strata volume (production channel, raw):',
      {s: float(NEWS_RAW_NAT[s][:, 3].sum()) for s in NEWS_STRATA})
print('share of days with any national signal:',
      round(float((NEWS_RAW_NAT['all'].sum(1) > 0).mean()), 4))
print('share of city-day cells with any signal:',
      round(float((NEWS_RAW_CITY['all'].sum(-1) > 0).mean()), 4))

In [ ]:
# === 3d. Policy-news fusion, and per-scenario array assembly ===
# The news ontology was built to mirror the policy taxonomy, so the two vocabularies line
# up directly. Only columns that are actually non-zero in the 22 used events are mapped;
# market_operation and production are all-zero in the regulation record and are left
# alone rather than given a spurious weight.
POLICY_NEWS_ALIGN = {
    'expected_price_pressure':  'pressure_net',     # both signed, same convention
    'chili':                    'chili_focus',
    'price_management':         'trade_policy',
    'import_export':            'trade_policy',
    'supply_reserve':           'supply_reserve',
    'procurement_distribution': 'supply_reserve',
    'subsidy_assistance':       'supply_reserve',
}

def policy_salience(P, nat):
    """Amplify each regulation by the news coverage it actually received.

    A regulation nobody reported on probably did not move a market; one that dominated
    the trade-policy channel for a week plausibly did. `nat` is z-scored on train, so
    relu(nat) is coverage ABOVE the training mean and the factor is >= 1: coverage can
    amplify a policy effect but never flip its sign or erase it.
    """
    out = P.copy()
    hit = []
    for j, col in enumerate(POLICY_COLS):
        ch = POLICY_NEWS_ALIGN.get(col)
        if ch is None:
            continue
        out[:, j] = P[:, j] * (1.0 + np.maximum(nat[:, NEWS_CHANNELS.index(ch)], 0.0))
        hit.append(col)
    return out.astype(np.float32), hit


def scenario_arrays(sc):
    """Return (Xnode, Xglobal, node_slices, global_slices, domain_names) for a scenario.

    Baseline arrays are never mutated; blocks are appended and their column ranges
    recorded so the context encoder can address them.

    Routing rule for national news: a block appended to Xglobal without being named in
    domain_names would reach the model ONLY through CMIL.head_gate - four scalar weights.
    So when news is not given its own expert it is folded into the economy expert's
    slice, where it is actually read. Otherwise 'news as a feature' would be a no-op by
    construction and S1 would test nothing.
    """
    xn, xg = XNODE_BASE, XGLOBAL_BASE
    ns = {k: list(v) for k, v in NODE_SLICES_BASE.items()}
    gs = {k: list(v) for k, v in GLOBAL_SLICES_BASE.items()}
    dom = list(DOMAINS_BASE)

    nat = None
    nw = sc.get('news')
    if nw:
        nat, cty = build_news(nw.get('stratum', 'all'),
                              nw.get('half_life'), nw.get('share_norm'))
        if nw.get('scope', 'both') in ('city', 'both'):
            n0 = xn.shape[-1]
            xn = np.concatenate([xn, cty], axis=-1).astype(np.float32)
            ns['news_node'] = list(range(n0, xn.shape[-1]))
    add_nat = bool(nw) and nw.get('scope', 'both') in ('national', 'both')

    pol = sc.get('policy')
    P = None
    if pol:
        P = G_POLICY_BASE
        if pol == 'salient':
            assert nat is not None, 'salience weighting needs the news block'
            P, _ = policy_salience(G_POLICY_BASE, nat)

    def _append(block, name, as_expert=True):
        nonlocal xg, gs, dom
        g0 = xg.shape[1]
        xg = np.concatenate([xg, block], axis=1).astype(np.float32)
        rng = list(range(g0, xg.shape[1]))
        if as_expert:
            gs[name] = rng
            dom = dom + [name]
        else:
            gs['economy'] = gs['economy'] + rng      # read by the economy expert
        return rng

    if sc.get('fuse_expert'):
        # S8 fused: ONE joint expert over [policy || news]. The two sources are seen
        # together rather than gated against each other.
        assert P is not None and add_nat, 'fuse_expert needs both policy and national news'
        _append(np.concatenate([P, nat], axis=1), 'polnews')
    else:
        if P is not None:
            _append(P, 'policy')
        if add_nat:
            _append(nat, 'news', as_expert=bool(sc.get('news_expert')))
    return xn, xg, ns, gs, dom


_probe = [('baseline', {}),
          ('S1 news (feature)', {'news': {'stratum': 'all'}}),
          ('S5 news (expert)', {'news': {'stratum': 'all'}, 'news_expert': True}),
          ('S8 policy only', {'policy': 'raw'}),
          ('S8 fused', {'news': {'stratum': 'all'}, 'policy': 'raw', 'fuse_expert': True}),
          ('S8 salience', {'news': {'stratum': 'all'}, 'policy': 'salient', 'fuse_expert': True})]
for _name, _sc in _probe:
    a, b, _, _, d = scenario_arrays(_sc)
    print(f'{_name:20s} Fn={a.shape[-1]:3d}  Fg={b.shape[1]:3d}  domains={d}')
_, _hit = policy_salience(G_POLICY_BASE, build_news('all')[0])
print(f'\nsalience-weighted policy columns ({len(_hit)} of {len(POLICY_COLS)}): {_hit}')

In [ ]:
# === 5. Model (Path B): Multi-scale Dilated Temporal Encoder + per-market Context Attribution ===
class DilatedTemporalEncoder(nn.Module):
    """Multi-scale dilated temporal encoder (Graph WaveNet philosophy, no adaptive graph).

    Per-node sequence -> stacked causal dilated conv blocks (gated activation, residual,
    skip) -> market representation at the final step. Replaces the LSTM market encoder.
    """
    def __init__(self, fn, d, dilations=(1, 2, 4, 8, 16), kernel=2):
        super().__init__()
        self.dilations, self.kernel = dilations, kernel
        self.input_proj = nn.Conv1d(fn, d, 1)
        self.filt = nn.ModuleList(nn.Conv1d(d, d, kernel, dilation=dl) for dl in dilations)
        self.gate = nn.ModuleList(nn.Conv1d(d, d, kernel, dilation=dl) for dl in dilations)
        self.resid = nn.ModuleList(nn.Conv1d(d, d, 1) for _ in dilations)
        self.skipc = nn.ModuleList(nn.Conv1d(d, d, 1) for _ in dilations)
        self.out = nn.Sequential(nn.ReLU(), nn.Conv1d(d, d, 1))

    def forward(self, x):                              # x: (B*N, L, Fn)
        x = self.input_proj(x.transpose(1, 2))         # (B*N, d, L)
        skip = 0
        for i, dl in enumerate(self.dilations):
            xp = F.pad(x, ((self.kernel - 1) * dl, 0))                 # causal left pad
            z = torch.tanh(self.filt[i](xp)) * torch.sigmoid(self.gate[i](xp))  # gated, (B*N,d,L)
            skip = skip + self.skipc[i](z)
            x = x + self.resid[i](z)                                   # residual
        return self.out(skip)[:, :, -1]                # (B*N, d) at last step


class ContextEncoder(nn.Module):
    """Five-domain context encoder: price, weather, supply, economy, and policy."""
    def __init__(self, fn, fg, dc, node_slices, global_slices, domain_names):
        super().__init__()
        self.node_slices, self.global_slices, self.domain_names = node_slices, global_slices, domain_names
        self.price = nn.Linear(len(node_slices['price']), dc)
        self.weather = nn.Linear(len(node_slices['weather']), dc)
        self.supply = nn.Linear(len(node_slices['supply']), dc)
        self.economy = nn.Linear(len(global_slices['economy']), dc)
        self.policy = nn.Linear(len(global_slices['policy']), dc)
        self.weight_head = nn.Sequential(nn.Linear(len(domain_names) * dc, dc), nn.ReLU(), nn.Linear(dc, len(domain_names)))

    def _node_summary(self, xn, key):
        return xn[..., self.node_slices[key]].mean(dim=(1, 2))

    def forward(self, xn, xg):
        ep = torch.tanh(self.price(self._node_summary(xn, 'price')))
        ew = torch.tanh(self.weather(self._node_summary(xn, 'weather')))
        es = torch.tanh(self.supply(self._node_summary(xn, 'supply')))
        ee = torch.tanh(self.economy(xg[:, self.global_slices['economy']]))
        e_policy = torch.tanh(self.policy(xg[:, self.global_slices['policy']]))
        E = torch.stack([ep, ew, es, ee, e_policy], dim=1)
        ctx_weight = F.softmax(self.weight_head(E.reshape(E.shape[0], -1)), dim=-1)
        C = (ctx_weight.unsqueeze(-1) * E).sum(dim=1)
        return C, ctx_weight, E

class NodeContextAttention(nn.Module):
    """Per-market context attribution. Each market forms a query from (H_i, C_t) and attends
    over the factor experts, so different markets weight environment / supply / economy /
    history differently (e.g. Surabaya -> economy, Banyuwangi -> weather). The attention is
    load-bearing: its output modulates the market representation."""
    def __init__(self, d, dc):
        super().__init__()
        self.q   = nn.Sequential(nn.Linear(d + dc, d), nn.ReLU(), nn.Linear(d, dc))
        self.key = nn.Linear(dc, dc)
        self.val = nn.Linear(dc, d)

    def forward(self, H, C, E):                        # H:(B,N,d) C:(B,dc) E:(B,5,dc)
        B, N, _ = H.shape
        q = self.q(torch.cat([H, C.unsqueeze(1).expand(-1, N, -1)], -1))  # (B,N,dc)
        K, V = self.key(E), self.val(E)                                   # (B,5,dc),(B,5,d)
        att = F.softmax(torch.einsum('bnd,bed->bne', q, K) / (q.shape[-1] ** 0.5), -1)  # (B,N,5)
        ctx = torch.einsum('bne,bed->bnd', att, V)                        # (B,N,d) per-market context
        return ctx, att


class CMIL(nn.Module):
    """City-level interaction graph (kept from the stable design): learnable city identities,
    context-weighted relation heads, alpha floor anchoring A_phys. The learned structure is
    stable across regimes -- that stability is now a reported finding, not a target to defeat."""
    def __init__(self, d, dc, R, Fg, n_heads=4, context_alpha=True):
        super().__init__()
        assert d % n_heads == 0, 'd must be divisible by n_heads'
        self.d, self.R, self.n_heads, self.dh = d, R, n_heads, d // n_heads
        self.context_alpha = context_alpha
        self.city_emb = nn.Parameter(torch.randn(R, d) * 0.1)
        self.gamma = nn.Linear(dc, d); self.beta = nn.Linear(dc, d)
        self.Wq = nn.Linear(d, d, bias=False)
        self.Wk = nn.Linear(d, d, bias=False)
        self.Wv = nn.Linear(d, d, bias=False)
        self.head_gate = nn.Linear(dc + Fg, n_heads)
        self.context_edge_bias = nn.Sequential(nn.Linear(dc, d), nn.Tanh(), nn.Linear(d, 1))
        if context_alpha:
            self.alpha_head = nn.Sequential(nn.Linear(dc, dc), nn.ReLU(), nn.Linear(dc, 1))
        else:
            self.alpha_logit = nn.Parameter(torch.tensor(0.0))
        self.alpha_min, self.alpha_span = 0.2, 0.6

    def forward(self, H, Sbar, C, Aphys, xg):
        B = H.shape[0]
        G  = torch.einsum('nr,bnd->brd', Sbar, H) + self.city_emb.unsqueeze(0)
        Gt = self.gamma(C).unsqueeze(1) * G + self.beta(C).unsqueeze(1)
        Q = self.Wq(Gt).view(B, self.R, self.n_heads, self.dh).transpose(1, 2)
        K = self.Wk(Gt).view(B, self.R, self.n_heads, self.dh).transpose(1, 2)
        V = self.Wv(G)
        Ih = torch.matmul(Q, K.transpose(-1, -2)) / (self.dh ** 0.5)
        head_w = F.softmax(self.head_gate(torch.cat([C, xg], dim=-1)), dim=-1)
        I = (head_w.view(B, self.n_heads, 1, 1) * Ih).sum(1)
        temp = 0.5 + 0.5 * torch.sigmoid(self.context_edge_bias(C).view(-1, 1, 1))
        Actx = F.softmax(I / temp, dim=-1)
        raw = (torch.sigmoid(self.alpha_head(C)).view(-1, 1, 1) if self.context_alpha
               else torch.sigmoid(self.alpha_logit))
        alpha = self.alpha_min + self.alpha_span * raw
        At = alpha * Aphys.unsqueeze(0) + (1.0 - alpha) * Actx
        return torch.matmul(At, V), At, {'Actx': Actx, 'alpha': alpha, 'temperature': temp, 'head_w': head_w}


class CMIN(nn.Module):
    def __init__(self, Fn, Fg, d, dc, H):
        super().__init__()
        self.enc       = DilatedTemporalEncoder(Fn, d)
        self.ctx       = ContextEncoder(Fn, Fg, dc, NODE_FEATURE_SLICES, GLOBAL_FEATURE_SLICES, DOMAIN_NAMES)
        self.node_attn = NodeContextAttention(d, dc)
        self.cmil      = CMIL(d, dc, R, Fg, n_heads=CFG.get('n_heads', 4),
                              context_alpha=CFG.get('context_alpha', True))
        self.gate = nn.Linear(2 * d, d)
        self.gate_min, self.gate_span = 0.15, 0.70
        self.dec  = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H))

    def forward(self, xn, xg, Sbar, S, Aphys):
        B, N, L, Fn = xn.shape
        h = self.enc(xn.reshape(B * N, L, Fn)).reshape(B, N, -1)          # dilated temporal encoder
        C, ctx_weight, E = self.ctx(xn, xg)
        node_ctx, node_att = self.node_attn(h, C, E)                      # per-market context attribution
        h = h + node_ctx                                                 # context modulates the market repr
        Gc, At, aux = self.cmil(h, Sbar, C, Aphys, xg)
        back = torch.einsum('nr,brd->bnd', S, Gc)
        g = self.gate_min + self.gate_span * torch.sigmoid(self.gate(torch.cat([h, back], -1)))
        z = g * h + (1.0 - g) * back
        aux['ctx_weight'] = ctx_weight                                   # global domain weights
        aux['node_ctx_attention'] = node_att                            # (B,N,5) per-market attribution
        aux['domain_names'] = DOMAIN_NAMES
        return self.dec(z), At, aux


print('Path-B model: dilated temporal encoder + per-market context attribution (stable interaction graph)')

In [ ]:
# === 5b. Scenario-aware model ===
# DilatedTemporalEncoder, NodeContextAttention and CMIL are reused unchanged from the
# production model above. Only the context encoder and the output head vary.

def seq_summary(xg_seq):
    """(B,L,Fg) -> mean, last, slope. S7 replaces the context snapshot with this."""
    B, Lw, _ = xg_seq.shape
    tt = torch.linspace(-1.0, 1.0, Lw, device=xg_seq.device).view(1, Lw, 1)
    mean = xg_seq.mean(1)
    last = xg_seq[:, -1]
    slope = (xg_seq * tt).mean(1) / (tt ** 2).mean()
    return mean, last, slope


class ContextEncoderS(nn.Module):
    """Context encoder whose experts are built FROM domain_names, not hard-coded.

    Each name in domain_names must resolve to either a node slice (price, weather,
    supply) or a global slice (economy, news, and policy when enabled). Dropping or
    adding a domain is then a data-side edit only - no surgery on this class. S7 feeds
    each global expert mean/last/slope over the window instead of the value at t,
    tripling its input width.
    """
    def __init__(self, dc, node_slices, global_slices, domain_names, ctx_seq=False):
        super().__init__()
        self.node_slices, self.global_slices = node_slices, global_slices
        self.domain_names, self.ctx_seq = list(domain_names), ctx_seq
        mult = 3 if ctx_seq else 1
        self.proj = nn.ModuleDict()
        for k in self.domain_names:
            if k in node_slices:
                self.proj[k] = nn.Linear(len(node_slices[k]), dc)
            elif k in global_slices:
                self.proj[k] = nn.Linear(mult * len(global_slices[k]), dc)
            else:
                raise KeyError(f'domain {k!r} has no node or global slice')
        self.weight_head = nn.Sequential(
            nn.Linear(len(self.domain_names) * dc, dc), nn.ReLU(),
            nn.Linear(dc, len(self.domain_names)))

    def _node_summary(self, xn, key):
        return xn[..., self.node_slices[key]].mean(dim=(1, 2))

    def _g(self, parts, key):
        return torch.cat([p[:, self.global_slices[key]] for p in parts], dim=-1)

    def forward(self, xn, xg_seq):
        parts = seq_summary(xg_seq) if self.ctx_seq else (xg_seq[:, -1],)
        E = []
        for k in self.domain_names:                      # order defines ctx_weight order
            src = self._node_summary(xn, k) if k in self.node_slices else self._g(parts, k)
            E.append(torch.tanh(self.proj[k](src)))
        E = torch.stack(E, dim=1)
        ctx_weight = F.softmax(self.weight_head(E.reshape(E.shape[0], -1)), dim=-1)
        C = (ctx_weight.unsqueeze(-1) * E).sum(dim=1)
        return C, ctx_weight, E


class CMINScenario(nn.Module):
    def __init__(self, Fn_, Fg_, d, dc, H_, node_slices, global_slices, domain_names, sc):
        super().__init__()
        self.sc = sc
        self.residual = bool(sc.get('residual_head'))
        self.spike = bool(sc.get('spike_head'))
        self.enc = DilatedTemporalEncoder(Fn_, d)
        self.ctx = ContextEncoderS(dc, node_slices, global_slices, domain_names,
                                   ctx_seq=bool(sc.get('ctx_seq')))
        self.node_attn = NodeContextAttention(d, dc)
        self.cmil = CMIL(d, dc, R, Fg_, n_heads=CFG.get('n_heads', 4),
                         context_alpha=CFG.get('context_alpha', True))
        self.gate = nn.Linear(2 * d, d)
        self.gate_min, self.gate_span = 0.15, 0.70
        self.dec = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H_))
        self.spike_head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H_)) if self.spike else None

    def forward(self, xn, xg_seq, Sbar, S_, Aphys):
        B, Nn, Lw, _ = xn.shape
        h = self.enc(xn.reshape(B * Nn, Lw, -1)).reshape(B, Nn, -1)
        C, ctx_weight, E = self.ctx(xn, xg_seq)
        node_ctx, node_att = self.node_attn(h, C, E)
        h = h + node_ctx
        Gc, At, aux = self.cmil(h, Sbar, C, Aphys, xg_seq[:, -1])
        back = torch.einsum('nr,brd->bnd', S_, Gc)
        g = self.gate_min + self.gate_span * torch.sigmoid(self.gate(torch.cat([h, back], -1)))
        z = g * h + (1.0 - g) * back

        out = self.dec(z)
        if self.residual:
            # S0: predict the DEVIATION from the last observed price. Without this the
            # model must reconstruct an absolute level from a 64-d bottleneck, which is
            # exactly the information a compressed representation has discarded - and is
            # why the level-predicting head loses to persistence at h=1.
            out = xn[:, :, -1, PRICE_IDX].unsqueeze(-1) + out

        aux['ctx_weight'] = ctx_weight
        aux['node_ctx_attention'] = node_att
        aux['domain_names'] = self.ctx.domain_names
        aux['spike_logits'] = self.spike_head(z) if self.spike else None
        return out, At, aux

print('scenario model ready')

In [ ]:
# === 5c. v3 model: expert representations exposed, REST-TS head, news ablation switch ===
class CMINv3(CMINScenario):
    """Adds three things to the v2 model:
      1. aux['E'] - per-sample expert representations, needed for effective rank.
         v2 stored only the batch-MEAN context weight, which averages away exactly the
         variation effective rank measures. That is why v2 cannot answer this offline.
      2. an optional REST-TS head: a news-only decoder supervised on the numerical
         residual, so no numerical pathway can reduce its loss.
      3. news_scale - multiplies the news block at inference, for the divergence test.
    """
    def __init__(self, Fn_, Fg_, d, dc, H_, node_slices, global_slices, domain_names, sc):
        super().__init__(Fn_, Fg_, d, dc, H_, node_slices, global_slices, domain_names, sc)
        self.rest = bool(sc.get('rest'))
        self.news_idx = domain_names.index('news') if 'news' in domain_names else None
        if self.rest:
            assert self.news_idx is not None, 'REST needs a news expert'
            self.dec_news = nn.Sequential(nn.Linear(dc, d), nn.ReLU(), nn.Linear(d, H_))

    def forward(self, xn, xg_seq, Sbar, S_, Aphys):
        out, At, aux = super().forward(xn, xg_seq, Sbar, S_, Aphys)
        _, _, E = self.ctx(xn, xg_seq)
        aux['E'] = E                                   # (B, n_experts, dc)
        if self.rest:
            e_news = E[:, self.news_idx, :]            # (B, dc) - news pathway ONLY
            pred_news = self.dec_news(e_news).unsqueeze(1).expand(-1, xn.shape[1], -1)
            aux['pred_num'] = out                      # numerical backbone forecast
            aux['pred_news'] = pred_news               # residual correction
            out = out + pred_news
        return out, At, aux


def zero_news(xn, xg, ns, gs):
    """Return copies with every news column set to 0 - the divergence probe."""
    xn2, xg2 = xn.clone(), xg.clone()
    if 'news' in gs:      xg2[..., gs['news']] = 0.0
    if 'polnews' in gs:   xg2[..., gs['polnews']] = 0.0
    if 'news_node' in ns: xn2[..., ns['news_node']] = 0.0
    return xn2, xg2


def effective_rank(Z, eps=1e-12):
    """Roy & Vetterli effective rank: exp(Shannon entropy of the normalised spectrum).

    High  -> representations spread out across inputs (content-dependent).
    ~1    -> every input maps to nearly the same direction (COLLAPSED).
    """
    Z = Z - Z.mean(0, keepdims=True)
    s = np.linalg.svd(Z, compute_uv=False)
    p = s / max(s.sum(), eps)
    p = p[p > eps]
    return float(np.exp(-(p * np.log(p)).sum()))

print('v3 model ready: aux[E] exposed, REST head available, news-zeroing probe defined')

In [ ]:
# === 4b. Windowing (scenario-aware) ===
L, H = CFG['L'], CFG['H']
REGIME_H = min(int(CFG.get('regime_horizon', 7)), H)
origins = list(range(L - 1, T - H))
def tgt_start(t): return dates[t + 1]
test_o = [t for t in origins if tgt_start(t) >= TEST_START]
pool   = [t for t in origins if tgt_start(t) <  TEST_START]
nval   = max(30, int(0.1 * len(pool)))
train_o, val_o = pool[:-nval], pool[-nval:]
print(f'windows -> train={len(train_o)}  val={len(val_o)}  test={len(test_o)}')

YY = torch.tensor(Y); OB = torch.tensor(obs); SPK = torch.tensor(SPIKE)
Sbar_t = torch.tensor(Sbar).to(DEVICE)
S_t    = torch.tensor(S).to(DEVICE)
Aphys_t = torch.tensor(A_phys).to(DEVICE)

class WDS(Dataset):
    """Always returns the global context as a WINDOW. Non-S7 scenarios take [:, -1],
    so a single dataset serves every scenario and the sampling can never differ."""
    def __init__(self, oo, xn, xg):
        self.oo = oo
        self.xn = torch.tensor(xn); self.xg = torch.tensor(xg)
    def __len__(self): return len(self.oo)
    def __getitem__(self, k):
        t = self.oo[k]
        return (self.xn[t - L + 1:t + 1].permute(1, 0, 2),   # (N,L,Fn)
                self.xg[t - L + 1:t + 1],                    # (L,Fg)
                YY[t + 1:t + 1 + H].permute(1, 0),           # (N,H)
                OB[t + 1:t + 1 + H].permute(1, 0),           # (N,H)
                SPK[t],                                      # (N,H)
                YY[t])                                       # (N,) persistence anchor

def make_loader_s(oo, sh, xn, xg):
    return DataLoader(WDS(oo, xn, xg), batch_size=CFG['batch'], shuffle=sh)

# Horizon weights for S0. h=1 errors are an order of magnitude smaller than h=30 errors,
# so a uniformly averaged loss spends almost none of its gradient on the short horizons.
_tr = np.array(train_o)
_scale = np.zeros(H, np.float64)
for h in range(1, H + 1):
    d_ = np.abs(Y[_tr + h] - Y[_tr]) * obs[_tr + h]
    _scale[h - 1] = d_.sum() / max(obs[_tr + h].sum(), 1)
HW = 1.0 / np.maximum(_scale, 1e-6)
HW = np.clip(HW / HW.mean(), 1.0 / SCFG['horizon_weight_clip'], SCFG['horizon_weight_clip'])
HW_t = torch.tensor(HW, dtype=torch.float32, device=DEVICE)
print('per-horizon |dY| scale  h1..h5:', np.round(_scale[:5], 4))
print('horizon weights         h1..h5:', np.round(HW[:5], 3), ' h30:', round(float(HW[-1]), 3))

In [ ]:
# === 6b. Train / evaluate, with per-seed retention and early-stop instrumentation ===
def masked_mae_w(p, y, m, w=None):
    e = torch.abs(p - y) * m
    if w is None:
        return e.sum() / m.sum().clamp(min=1)
    w = w.view(1, 1, -1)
    return (e * w).sum() / (m * w).sum().clamp(min=1)

def graph_sparsity_penalty_s(aux):
    lam = float(CFG.get('lambda_graph_sparsity', 0.03))
    if not lam or 'Actx' not in aux:
        return 0.0
    A = aux['Actx'].clamp_min(1e-9)
    return lam * (-(A * A.log()).sum(dim=-1).mean())

def train_scenario(sc, seed, epochs=None, patience=None, verbose=False):
    """Returns (model, arrays, info). `info` records where early stopping landed, so a
    run that hit the epoch cap - i.e. was still improving when cut off - is visible
    rather than silently truncated."""
    epochs = int(epochs or SCFG['epochs']); patience = int(patience or SCFG['patience'])
    xn, xg, ns, gs, dom = scenario_arrays(sc)
    torch.manual_seed(seed); np.random.seed(seed)
    model = CMINScenario(xn.shape[-1], xg.shape[1], CFG['d'], CFG['d_ctx'], H,
                         ns, gs, dom, sc).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=min(CFG['lr'], 5e-4),
                           weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
    tl = make_loader_s(train_o, True, xn, xg)
    vl = make_loader_s(val_o, False, xn, xg)
    hw = HW_t if sc.get('horizon_weight') else None
    lam_spk = float(SCFG['lambda_spike']) if sc.get('spike_head') else 0.0

    best, best_state, best_ep, wait, hist = 1e9, None, 0, 0, []
    ep = 0
    for ep in range(epochs):
        model.train()
        for xnb, xgb, y, m, spk, _ in tl:
            xnb, xgb = xnb.to(DEVICE), xgb.to(DEVICE)
            y, m, spk = y.to(DEVICE), m.to(DEVICE), spk.to(DEVICE)
            opt.zero_grad()
            p, _, aux = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
            loss = masked_mae_w(p, y, m, hw) + graph_sparsity_penalty_s(aux)
            if lam_spk and aux['spike_logits'] is not None:
                bce = F.binary_cross_entropy_with_logits(aux['spike_logits'], spk, reduction='none')
                loss = loss + lam_spk * ((bce * m).sum() / m.sum().clamp(min=1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()

        model.eval(); vs = vn = 0.0
        with torch.no_grad():
            for xnb, xgb, y, m, spk, _ in vl:
                xnb, xgb, y, m = xnb.to(DEVICE), xgb.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
                p, _, _ = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
                vs += (torch.abs(p - y) * m).sum().item(); vn += m.sum().item()
        v = vs / max(vn, 1); sched.step(v); hist.append(v)
        if verbose:
            print(f'    ep {ep+1:02d} val={v:.4f}')
        if v < best - 1e-4:
            best, best_ep, wait = v, ep + 1, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    info = dict(best_epoch=best_ep, epochs_run=ep + 1, cap=epochs,
                capped=bool(ep + 1 >= epochs), best_val=best, history=hist)
    return model, (xn, xg), info

@torch.no_grad()
def predict_scenario(model, oo, xn, xg):
    model.eval(); P=[]; Yt=[]; Mt=[]; Sp=[]; Lg=[]; An=[]
    for xnb, xgb, y, m, spk, anchor in make_loader_s(oo, False, xn, xg):
        p, _, aux = model(xnb.to(DEVICE), xgb.to(DEVICE), Sbar_t, S_t, Aphys_t)
        P.append(p.cpu()); Yt.append(y); Mt.append(m); Sp.append(spk); An.append(anchor)
        Lg.append(aux['spike_logits'].cpu() if aux['spike_logits'] is not None else torch.zeros_like(p.cpu()))
    return (torch.cat(P), torch.cat(Yt), torch.cat(Mt), torch.cat(Sp),
            torch.cat(Lg), torch.cat(An))

def to_rp(z):
    return torch.exp(z * sd_t.view(1, -1, 1) + mu_t.view(1, -1, 1))

def eval_scenario(model, oo, xn, xg):
    P, Yv, Mv, Spk, Lg, An = predict_scenario(model, oo, xn, xg)
    Prp, Yrp = to_rp(P), to_rp(Yv)
    Perp = to_rp(An.unsqueeze(-1).expand(-1, -1, CFG['H']))
    rows, errs = [], {}
    for h in CFG['HORIZONS']:
        i = h - 1
        mm = Mv[:, :, i] > 0
        pe, ye, pp = Prp[:, :, i][mm], Yrp[:, :, i][mm], Perp[:, :, i][mm]
        sp = Spk[:, :, i][mm] > 0
        ae, ap = torch.abs(pe - ye), torch.abs(pp - ye)
        row = dict(h=h, MAE_Rp=float(ae.mean()),
                   RMSE_Rp=float(torch.sqrt(((pe - ye) ** 2).mean())),
                   MAPE_pct=float((ae / ye.clamp(min=1)).mean() * 100),
                   MAE_persistence=float(ap.mean()), n=int(mm.sum()))
        row['skill_vs_persistence_pct'] = 100.0 * (1.0 - row['MAE_Rp'] / max(row['MAE_persistence'], 1e-9))
        if sp.any():
            row['MAE_spike'] = float(ae[sp].mean())
            row['spike_n'] = int(sp.sum())
        if model.spike:
            pr = torch.sigmoid(Lg[:, :, i][mm]); y01 = sp.float()
            if 0 < y01.mean() < 1:
                order = torch.argsort(pr)
                ranks = torch.empty_like(pr); ranks[order] = torch.arange(len(pr), dtype=pr.dtype) + 1
                npos, nneg = y01.sum(), (1 - y01).sum()
                row['spike_AUC'] = float((ranks[y01 > 0].sum() - npos * (npos + 1) / 2) / (npos * nneg))
        rows.append(row)
        # per-horizon (window, node) absolute errors, retained per seed
        errs[h] = (torch.abs(Prp[:, :, i] - Yrp[:, :, i]) * (Mv[:, :, i] > 0)).numpy()
    return pd.DataFrame(rows).set_index('h'), errs

print('harness ready (per-seed retention + early-stop instrumentation)')

In [ ]:
# === 6b. Training with HORIZON-NORMALISED selection ===
def masked_mae_w(p, y, m, w=None):
    e = torch.abs(p - y) * m
    if w is None:
        return e.sum() / m.sum().clamp(min=1)
    w = w.view(1, 1, -1)
    return (e * w).sum() / (m * w).sum().clamp(min=1)

def graph_sparsity_penalty_s(aux):
    lam = float(CFG.get('lambda_graph_sparsity', 0.03))
    if not lam or 'Actx' not in aux: return 0.0
    A = aux['Actx'].clamp_min(1e-9)
    return lam * (-(A * A.log()).sum(dim=-1).mean())

# per-horizon error scale from the TRAIN split - the same vector that defines HW
HSCALE = torch.tensor(np.maximum(_scale, 1e-6), dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def val_score(model, loader):
    """Horizon-NORMALISED validation MAE.

    v2 averaged raw MAE over H=30. Long-horizon errors are ~10x larger, so selection was
    effectively h=30-only and fired at epochs arbitrary for h=1 - producing an ablation
    table that ranked by stopping epoch (corr +0.818). Dividing each horizon by its own
    train-residual scale gives every horizon equal say.
    """
    num = torch.zeros(H, device=DEVICE); den = torch.zeros(H, device=DEVICE)
    for xnb, xgb, y, m, spk, _ in loader:
        xnb, xgb, y, m = xnb.to(DEVICE), xgb.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
        p, _, _ = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
        num += ((p - y).abs() * m).sum(dim=(0, 1)); den += m.sum(dim=(0, 1)).clamp(min=1)
    return float(((num / den) / HSCALE).mean())

def train_scenario(sc, seed, epochs=None, patience=None, verbose=False):
    epochs = int(epochs or SCFG['epochs']); patience = int(patience or SCFG['patience'])
    xn, xg, ns, gs, dom = scenario_arrays(sc)
    torch.manual_seed(seed); np.random.seed(seed)
    model = CMINv3(xn.shape[-1], xg.shape[1], CFG['d'], CFG['d_ctx'], H, ns, gs, dom, sc).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=min(CFG['lr'], 5e-4), weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
    tl = make_loader_s(train_o, True, xn, xg); vl = make_loader_s(val_o, False, xn, xg)
    hw = HW_t if sc.get('horizon_weight') else None
    lam_rest = float(CFG.get('lambda_rest', 1.0)) if sc.get('rest') else 0.0

    best, best_state, best_ep, wait, hist = 1e9, None, 0, 0, []
    ep = 0
    for ep in range(epochs):
        model.train()
        for xnb, xgb, y, m, spk, _ in tl:
            xnb, xgb = xnb.to(DEVICE), xgb.to(DEVICE)
            y, m = y.to(DEVICE), m.to(DEVICE)
            opt.zero_grad()
            p, _, aux = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
            if lam_rest:
                # REST-TS: the numerical backbone is supervised on y; the news branch is
                # supervised ONLY on the residual it leaves behind. Detaching the residual
                # means no numerical pathway can reduce this term - the news branch has to
                # find content in the text or contribute nothing.
                pn = aux['pred_num']
                loss = masked_mae_w(pn, y, m, hw)
                loss = loss + lam_rest * masked_mae_w(aux['pred_news'], (y - pn).detach(), m, hw)
            else:
                loss = masked_mae_w(p, y, m, hw)
            loss = loss + graph_sparsity_penalty_s(aux)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()

        model.eval(); v = val_score(model, vl); sched.step(v); hist.append(v)
        if verbose: print(f'    ep {ep+1:02d} val_norm={v:.4f}')
        if v < best - 1e-5:
            best, best_ep, wait = v, ep + 1, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    return model, (xn, xg, ns, gs, dom), dict(best_epoch=best_ep, epochs_run=ep + 1,
                                              capped=bool(ep + 1 >= epochs),
                                              best_val=best, history=hist)

@torch.no_grad()
def instrument(model, arrays, oo):
    """effective rank per expert + divergence delta from zeroing news on the SAME model."""
    xn, xg, ns, gs, dom = arrays
    model.eval()
    Es, Pf, Pz, Y, M = [], [], [], [], []
    for xnb, xgb, y, m, spk, _ in make_loader_s(oo, False, xn, xg):
        xnb, xgb = xnb.to(DEVICE), xgb.to(DEVICE)
        p, _, aux = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
        Es.append(aux['E'].cpu().numpy()); Pf.append(p.cpu()); Y.append(y); M.append(m)
        xz, gz = zero_news(xnb, xgb, ns, gs)
        pz, _, _ = model(xz, gz, Sbar_t, S_t, Aphys_t)
        Pz.append(pz.cpu())
    E = np.concatenate(Es, 0)                       # (N, n_experts, dc)
    er = {d: effective_rank(E[:, i, :]) for i, d in enumerate(dom)}
    Pf, Pz = torch.cat(Pf), torch.cat(Pz)
    Yv, Mv = torch.cat(Y), torch.cat(M)
    Prp, Zrp, Yrp = to_rp(Pf), to_rp(Pz), to_rp(Yv)
    dl = {}
    for h in CFG['HORIZONS']:
        i = h - 1; mm = Mv[:, :, i] > 0
        num = torch.sqrt((((Prp[:, :, i] - Zrp[:, :, i]) ** 2)[mm]).mean())
        den = torch.abs(Prp[:, :, i] - Yrp[:, :, i])[mm].mean()
        dl[h] = float(num / den.clamp(min=1e-9))
    return er, dl

print('horizon-normalised selection active; instrumentation ready')

In [ ]:
# === 7c. Placebo transforms (v4: any auxiliary block, not just news) ===
# v3 scrambled only the news columns, so policy could never be placebo-tested. Here the
# target is explicit, and defaults to every auxiliary block present.
_rng_placebo = np.random.default_rng(12345)
AUX_GLOBAL = ('news', 'policy', 'polnews')
AUX_NODE = ('news_node',)

def _targets(sc, ns, gs):
    want = sc.get('placebo_on')
    want = (want,) if isinstance(want, str) else (want or AUX_GLOBAL + AUX_NODE)
    return ([k for k in AUX_GLOBAL if k in gs and k in want],
            [k for k in AUX_NODE if k in ns and k in want])

def apply_placebo(sc, xn, xg, ns, gs):
    kind = sc.get('placebo')
    if not kind:
        return xn, xg
    gk, nk = _targets(sc, ns, gs)
    xn, xg = xn.copy(), xg.copy()
    if kind == 'time':
        # break the block's correspondence to days; volume and marginals preserved
        perm = _rng_placebo.permutation(xg.shape[0])
        for k in gk: xg[:, gs[k]] = xg[perm][:, gs[k]]
        for k in nk: xn[..., ns[k]] = xn[perm][..., ns[k]]
    elif kind == 'city':
        # keep the timing, scramble WHICH market receives each story
        perm = _rng_placebo.permutation(xn.shape[1])
        for k in nk: xn[..., ns[k]] = xn[:, perm][..., ns[k]]
        if not nk:
            print('  [!] city placebo requested but this scenario has no node-level block')
    return xn, xg

_orig_scenario_arrays = scenario_arrays
def scenario_arrays(sc):                       # noqa: F811 - deliberate wrapper
    xn, xg, ns, gs, dom = _orig_scenario_arrays(sc)
    xn, xg = apply_placebo(sc, xn, xg, ns, gs)
    return xn, xg, ns, gs, dom

print('placebo wrapper installed (targets:', AUX_GLOBAL + AUX_NODE, ')')

In [ ]:
# === P1. Ablation ladder: graph mode as an explicit switch ===
# The ladder needs adjacency modes that CMIL does not expose, so they are added by subclass
# here rather than by editing the copied v4 cells. Everything else -- temporal encoder,
# context encoder, fusion gate, head, loss -- is untouched, which is the whole point: the
# rungs must differ in the graph and in nothing else.
SMOKE = False         # True: 1 seed, 3 epochs, every model. Set False for the real run.

PROTO_DIR = os.path.join(OUT_DIR, 'protocol')
os.makedirs(PROTO_DIR, exist_ok=True)
TEST_ORIGINS = test_o                     # fixed here so every later cell scores the same set
SEEDS_N = 1 if SMOKE else SCFG['seeds']
EPOCHS  = 3 if SMOKE else SCFG['epochs']
print('output   ->', PROTO_DIR)
print('test origins:', len(TEST_ORIGINS))

GRAPH_MODES = ('none', 'phys', 'static', 'context')

class CMILModes(CMIL):
    """CMIL with the adjacency source switchable.

      none    : no message passing at all (encoder-only rung)
      phys    : A_t = A_phys, fixed for all t
      static  : A_t = softmax(W), learned but context-independent and time-invariant
      context : unchanged v4 behaviour

    A price-conditioned graph is NOT a mode here. It is obtained by feeding price-only
    context (ctx='price') into the unchanged context mode, which is the same thing with
    fewer moving parts and no masking of an already-pooled context vector.
    """
    def __init__(self, *a, graph_mode='context', **kw):
        super().__init__(*a, **kw)
        assert graph_mode in GRAPH_MODES, graph_mode
        self.graph_mode = graph_mode
        if graph_mode == 'static':
            self.static_logits = nn.Parameter(torch.randn(self.R, self.R) * 0.1)

    def forward(self, H, Sbar, C, Aphys, xg):
        if self.graph_mode == 'context':
            return super().forward(H, Sbar, C, Aphys, xg)

        B = H.shape[0]
        G = torch.einsum('nr,bnd->brd', Sbar, H) + self.city_emb.unsqueeze(0)
        if self.graph_mode == 'none':
            # Return the city representations untouched; the fusion gate in CMINScenario
            # then decides how much of a no-op neighbourhood to use.
            eye = torch.eye(self.R, device=H.device).unsqueeze(0).expand(B, -1, -1)
            return G, eye, {'Actx': eye, 'alpha': torch.ones(B, 1, 1, device=H.device),
                            'temperature': torch.ones(B, 1, 1, device=H.device),
                            'head_w': torch.zeros(B, self.n_heads, device=H.device)}

        Cg = C
        if self.graph_mode == 'phys':
            At = Aphys.unsqueeze(0).expand(B, -1, -1)
            V = self.Wv(G)
            aux = {'Actx': At, 'alpha': torch.ones(B, 1, 1, device=H.device),
                   'temperature': torch.ones(B, 1, 1, device=H.device),
                   'head_w': torch.zeros(B, self.n_heads, device=H.device)}
            return torch.matmul(At, V), At, aux

        # only 'static' reaches here
        Actx = F.softmax(self.static_logits, dim=-1).unsqueeze(0).expand(B, -1, -1)

        raw = (torch.sigmoid(self.alpha_head(Cg)).view(-1, 1, 1) if self.context_alpha
               else torch.sigmoid(self.alpha_logit))
        alpha = self.alpha_min + self.alpha_span * raw
        At = alpha * Aphys.unsqueeze(0) + (1.0 - alpha) * Actx
        V = self.Wv(G)
        return torch.matmul(At, V), At, {'Actx': Actx, 'alpha': alpha,
                                         'temperature': torch.ones(B, 1, 1, device=H.device),
                                         'head_w': torch.zeros(B, self.n_heads, device=H.device)}


# The nine rungs. `ctx` selects which context reaches the model at all; `graph` selects what
# shapes the adjacency. Rungs 5 and 6 differ ONLY in `graph`, which is the decisive contrast.
LADDER = {
    'L1_encoder_only':      dict(ctx='price', graph='none',    anchor=True),
    'L2_fixed_physical':    dict(ctx='price', graph='phys',    anchor=True),
    'L3_learned_static':    dict(ctx='price', graph='static',  anchor=True),
    'L4_price_dynamic':     dict(ctx='price', graph='context', anchor=True),
    'L5_ctx_features_only': dict(ctx='full',  graph='phys',    anchor=True),   # <-- control
    'L6_ctx_conditioned':   dict(ctx='full',  graph='context', anchor=True),   # <-- claim
    'L7_ctx_shuffled':      dict(ctx='full',  graph='context', anchor=True, placebo='time'),
    'L8_no_phys_anchor':    dict(ctx='full',  graph='context', anchor=False),
    # L9 (market-level vs city-level graph) is deliberately omitted. The protocol admits it
    # "only if both can be implemented without changing other components", and a market-level
    # graph replaces the pooling and unpooling stages -- so it would not be the same model
    # with a different graph. Report it as not run rather than as a like-for-like rung.
}
# Every rung is run under BOTH forecast heads. v4's Table 9 established that the residual
# parameterisation owns h <= 14 and the standard one owns h >= 14, so a ladder fixed to one
# head would be judged in the regime where that head is known to be weak -- which is exactly
# how the smoke run ended up showing a plain STGCN beating every rung at h=30.
HEADS = {'R': dict(residual_head=True),      # persistence-anchored: strong at short h
         'S': dict(residual_head=False)}     # standard decoder: strong at long h

DECISIVE = ('L5_ctx_features_only', 'L6_ctx_conditioned')

def rung_name(base, head):
    return '%s@%s' % (base, head)

print('ladder rungs:', len(LADDER))
print('decisive contrast: %s  vs  %s' % DECISIVE)
print('SMOKE =', SMOKE, '| seeds =', 1 if SMOKE else SCFG['seeds'],
      '| epochs =', 3 if SMOKE else SCFG['epochs'])

In [ ]:
# === H0b. Which Drive, which folder, and what is already in it ===
# Switching to a different Google account needs NO code change. drive.mount() in cell 1
# authorises whichever account you sign in as, and MyDrive resolves to that account's Drive,
# so these same paths point somewhere else entirely.
#
# What changes is what is already there, and that changes the runtime by a factor of six.
RUN_TAG = ''      # '' = the standard folder. Set e.g. 'run2' to isolate this run's outputs.

if RUN_TAG:
    PROTO_DIR = os.path.join(OUT_DIR, 'protocol_%s' % RUN_TAG)
    os.makedirs(PROTO_DIR, exist_ok=True)

_have = [f for f in os.listdir(PROTO_DIR) if f.endswith('.pkl')] if os.path.isdir(PROTO_DIR) else []
print('reads and writes : %s' % PROTO_DIR)
print('runs present     : %d' % len(_have))
print('device           : %s' % DEVICE)
if not _have:
    print('')
    print('  EMPTY DIRECTORY -- nothing will be reused, so this is the FULL suite:')
    print('    ladder      8 rungs x 2 heads x 10 seeds  = 160 runs')
    print('    baselines   naive/ML/temporal/graph        ~ 90 runs')
    print('    hierarchy  13 rungs x heads x 10 seeds     ~ 180 runs')
    print('    capacity    6 matched baselines x 10 seeds =  60 runs')
    print('  Roughly 13 h on a T4, resumable per (model, seed). Not feasible on CPU.')
    print('')
    print('  If you wanted the ~2 h follow-up instead, stop here and copy the other')
    print('  account\'s MyDrive/cmin_v4_out/protocol/ into this account first. Sharing the')
    print('  folder and using "Add shortcut to Drive" is enough; the files are read-only')
    print('  to this notebook until it writes new rungs.')
else:
    print('  these will be reused; only missing runs train.')

In [ ]:
# === P2. Binding the ladder to the v4 machinery ===
# Every rung shares one temporal encoder, one context encoder, one fusion gate and one head.
# What differs is (a) which context reaches the model and (b) what shapes the adjacency.
# Nothing else may vary, or the comparison stops isolating the mechanism.
LADDER_SC = dict(residual_head=True, horizon_weight=True)

def ladder_arrays(cfg):
    """Arrays for one rung.

    ctx='price' zeroes the non-price node blocks and the whole global block instead of
    deleting them. Deleting would change Fn/Fg and therefore parameter count, which would
    confound 'less context' with 'smaller model' -- the exact confound this ladder exists
    to remove.
    """
    xn, xg, ns, gs, dom = scenario_arrays(dict(LADDER_SC))
    if cfg['ctx'] == 'price':
        xn = xn.copy(); xg = np.zeros_like(xg)
        for k, idx in ns.items():
            if k != 'price':
                xn[..., list(idx)] = 0.0
    if cfg.get('placebo') == 'time':
        # v4's apply_placebo only routes ('news','policy','polnews','news_node'). Asking it
        # for 'economy' was a silent no-op, so L7 came out BYTE-IDENTICAL to L6 --
        # 2185/4867/7886/12184/19319 at every horizon. Shift the context here instead.
        # A circular shift, not a permutation: it preserves each channel's autocorrelation
        # and marginals exactly and destroys only the alignment to price, which is the one
        # thing this control is meant to remove.
        off = int(0.37 * xg.shape[0])
        xn = xn.copy()
        xg = np.roll(xg, off, axis=0)
        for k, idx in ns.items():
            if k != 'price':
                cols = list(idx)
                xn[..., cols] = np.roll(xn[..., cols], off, axis=0)
    return xn, xg, ns, gs, dom

def build_ladder_model(cfg, head='R'):
    xn, xg, ns, gs, dom = ladder_arrays(cfg)
    sc = dict(LADDER_SC); sc.update(HEADS[head])
    m = CMINScenario(xn.shape[-1], xg.shape[-1], CFG['d'], CFG['d_ctx'], CFG['H'],
                     ns, gs, dom, sc)
    m.cmil = CMILModes(CFG['d'], CFG['d_ctx'], R, xg.shape[-1],
                       n_heads=CFG.get('n_heads', 4),
                       context_alpha=CFG.get('context_alpha', True),
                       graph_mode=cfg['graph'])
    if not cfg.get('anchor', True):
        # alpha pinned to 0 -> the physical graph contributes nothing and A_t is purely learned
        m.cmil.alpha_min, m.cmil.alpha_span = 0.0, 0.0
    return m.to(DEVICE), (xn, xg)

def rung_ckpt(name, seed):
    return os.path.join(PROTO_DIR, 'w_%s__s%d.pt' % (name, seed))

def train_ladder_rung(cfg, seed, epochs=None, name=None, head='R'):
    torch.manual_seed(seed); np.random.seed(seed)
    epochs = epochs or SCFG['epochs']
    model, (xn, xg) = build_ladder_model(cfg, head)
    opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'],
                           weight_decay=CFG['weight_decay'])
    dtr = make_loader_s(train_o, True,  xn, xg)
    dva = make_loader_s(val_o,   False, xn, xg)
    dte = make_loader_s(TEST_ORIGINS, False, xn, xg)

    def fwd(b):
        xnb, xgb, y, ob, spk, anc = [z.to(DEVICE) for z in b]
        out = model(xnb, xgb, Sbar_t, S_t, Aphys_t)
        p = out[0] if isinstance(out, (tuple, list)) else out
        return p, y, ob

    def score(dl):
        model.eval(); num = den = 0.0
        with torch.no_grad():
            for b in dl:
                p, y, ob = fwd(b)
                w = HW_t.view(1, 1, -1)
                num += ((p - y).abs() * ob * w).sum().item()
                den += (ob * w).sum().item()
        return num / max(den, 1e-9)

    best, bad, bstate = np.inf, 0, None
    for ep in range(epochs):
        model.train()
        for b in dtr:
            p, y, ob = fwd(b)
            loss = ((p - y).abs() * ob * HW_t.view(1, 1, -1)).sum() / (
                (ob * HW_t.view(1, 1, -1)).sum().clamp(min=1e-9))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()
        v = score(dva)
        if v < best - 1e-6:
            best, bad = v, 0
            bstate = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= (SCFG['patience'] if epochs > 5 else 2):
                break
    if bstate:
        model.load_state_dict(bstate)
        if name:
            torch.save(bstate, rung_ckpt(name, seed))

    preds = []
    model.eval()
    with torch.no_grad():
        for b in dte:
            p, _, _ = fwd(b)
            preds.append(p.cpu().numpy())
    P = np.concatenate(preds, 0)
    assert P.shape == (len(TEST_ORIGINS), Y.shape[1], CFG['H']), P.shape
    return P

def load_rung_weights(model, name, seed):
    f = rung_ckpt(name, seed)
    if os.path.exists(f):
        model.load_state_dict(torch.load(f, map_location=DEVICE)); return True
    return False

print('ladder glue ready: ladder_arrays / build_ladder_model / train_ladder_rung')

In [ ]:
# === P3. Baseline families (protocol Table 4) ===
# Registered independently so a failure in one records a reason and the suite continues.
# The statistical ones are exact; the neural ones are compact re-implementations matched on
# encoder width, optimiser, budget and selection rule -- not the authors' released code.
import traceback

BASELINES = {}
def baseline(name, family):
    def deco(fn):
        BASELINES[name] = dict(fn=fn, family=family)
        return fn
    return deco

def _yhat_shape(oo):
    return (len(oo), Y.shape[1], CFG['H'])

# ---------------------------------------------------------------- naive / statistical ---
@baseline('Persistence', 'naive')
def _persist(tr, te):
    return np.stack([np.repeat(Y[t][:, None], CFG['H'], 1) for t in te])

@baseline('SeasonalNaive7', 'naive')
def _snaive(tr, te):
    out = np.zeros(_yhat_shape(te))
    for k, t in enumerate(te):
        for h in range(1, CFG['H'] + 1):
            out[k, :, h - 1] = Y[t - 7 + ((h - 1) % 7) + 1]
    return out

@baseline('Drift', 'naive')
def _drift(tr, te):
    out = np.zeros(_yhat_shape(te))
    for k, t in enumerate(te):
        slope = (Y[t] - Y[t - 30]) / 30.0
        for h in range(1, CFG['H'] + 1):
            out[k, :, h - 1] = Y[t] + slope * h
    return out

@baseline('ETS', 'statistical')
def _ets(tr, te):
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    end = min(te)
    out = np.zeros(_yhat_shape(te))
    fc = np.zeros((Y.shape[1], CFG['H']))
    for n in range(Y.shape[1]):
        s = pd.Series(Y[:end, n]).ffill().bfill()
        try:
            f = ExponentialSmoothing(s, trend='add', damped_trend=True,
                                     seasonal=None).fit(optimized=True)
            fc[n] = f.forecast(CFG['H']).to_numpy()
        except Exception:
            fc[n] = s.iloc[-1]
    # ETS is refit once on the training portion; re-fitting per origin over 120 markets
    # and hundreds of origins is not affordable and would not change the ranking.
    for k, t in enumerate(te):
        out[k] = Y[t][:, None] + (fc - fc[:, :1])
    return out

# ------------------------------------------------------------------- machine learning ---
@baseline('XGBoost', 'ml')
def _xgb(tr, te):
    from xgboost import XGBRegressor
    lags = [1, 2, 3, 5, 7, 14, 30]
    def feats(origins):
        F_ = [np.stack([Y[np.array(origins) - L] for L in lags], -1)]      # (T,N,lags)
        g = Xglobal[np.array(origins)]                                     # (T,Fg)
        F_.append(np.repeat(g[:, None, :], Y.shape[1], 1))
        return np.concatenate(F_, -1)
    Xtr, Xte = feats(tr), feats(te)
    out = np.zeros(_yhat_shape(te))
    for h in range(1, CFG['H'] + 1):
        ytr = np.stack([Y[np.array(tr) + h]])[0]
        m = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8, n_jobs=4,
                         tree_method='hist', verbosity=0)
        m.fit(Xtr.reshape(-1, Xtr.shape[-1]), ytr.reshape(-1))
        out[:, :, h - 1] = m.predict(Xte.reshape(-1, Xte.shape[-1])).reshape(len(te), -1)
    return out

# --------------------------------------------------------------------- temporal neural --
class _Seq(nn.Module):
    def __init__(self, kind, Fn, d, H):
        super().__init__()
        self.kind = kind
        if kind in ('lstm', 'gru'):
            self.rnn = (nn.LSTM if kind == 'lstm' else nn.GRU)(Fn, d, batch_first=True)
        else:                                            # tcn
            self.rnn = DilatedTemporalEncoder(Fn, d)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H))
    def forward(self, xn, xg=None, *a):
        B, N, L_, Fn = xn.shape
        x = xn.reshape(B * N, L_, Fn)
        z = self.rnn(x) if self.kind == 'tcn' else self.rnn(x)[0][:, -1]
        return self.head(z).reshape(B, N, -1)

class _NHiTS(nn.Module):
    """Compact N-HiTS: pooled multi-rate MLP stacks with residual subtraction."""
    def __init__(self, Fn, d, H, rates=(1, 2, 4)):
        super().__init__()
        self.rates = rates
        self.blocks = nn.ModuleList()
        for r in rates:
            self.blocks.append(nn.Sequential(
                nn.Linear(max(1, CFG['L'] // r), d), nn.ReLU(),
                nn.Linear(d, d), nn.ReLU(), nn.Linear(d, CFG['L'] + H)))
    def forward(self, xn, xg=None, *a):
        B, N, L_, _ = xn.shape
        res = xn[..., 0].reshape(B * N, L_)
        out = 0.0
        for r, blk in zip(self.rates, self.blocks):
            p = F.avg_pool1d(res.unsqueeze(1), r).squeeze(1) if r > 1 else res
            z = blk(p)
            back, fore = z[:, :L_], z[:, L_:]
            res = res - back
            out = out + fore
        return out.reshape(B, N, -1)

# ------------------------------------------------------------------------ graph neural --
class _STGCN(nn.Module):
    """Fixed-graph STGNN: temporal encoder + propagation over A_phys only."""
    def __init__(self, Fn, d, H, A):
        super().__init__()
        self.enc = DilatedTemporalEncoder(Fn, d)
        self.register_buffer('A', A)
        self.gc = nn.Linear(d, d)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H))
    def forward(self, xn, xg=None, *a):
        B, N, L_, Fn = xn.shape
        h = self.enc(xn.reshape(B * N, L_, Fn)).reshape(B, N, -1)
        h = h + torch.relu(self.gc(torch.einsum('ij,bjd->bid', self.A, h)))
        return self.head(h)

class _AdaptiveGraph(nn.Module):
    """Adaptive-graph family (AGCRN / Graph WaveNet): adjacency from learned node
    embeddings, context-independent by construction. `mode` only changes the temporal
    backbone -- gru for AGCRN-style, tcn for Graph WaveNet-style."""
    def __init__(self, Fn, d, H, N, mode='tcn', emb=16):
        super().__init__()
        self.mode = mode
        self.E = nn.Parameter(torch.randn(N, emb) * 0.1)
        self.enc = (DilatedTemporalEncoder(Fn, d) if mode == 'tcn'
                    else nn.GRU(Fn, d, batch_first=True))
        self.gc = nn.Linear(d, d)
        self.head = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, H))
    def forward(self, xn, xg=None, *a):
        B, N, L_, Fn = xn.shape
        x = xn.reshape(B * N, L_, Fn)
        h = (self.enc(x) if self.mode == 'tcn' else self.enc(x)[0][:, -1]).reshape(B, N, -1)
        A = F.softmax(torch.relu(self.E @ self.E.t()), dim=-1)
        h = h + torch.relu(self.gc(torch.einsum('ij,bjd->bid', A, h)))
        return self.head(h)

print('registered %d statistical/ML baselines' % len(BASELINES))
print('neural baselines: LSTM, GRU, TCN, N-HiTS, STGCN(A_phys), AGCRN-style, GraphWaveNet-style')

In [ ]:
# === P4. Register the neural and graph baselines ===
# The previous cell only DEFINES the model classes. Without this cell they are never
# trained, and Table 4's temporal-neural, fixed-graph and adaptive-graph rows stay empty --
# which is precisely the gap the protocol was written to close. The smoke run caught it:
# five naive/ML rows, and nothing from any neural family.
BASE_XN, BASE_XG, _ns, _gs, _dom = ladder_arrays(dict(ctx='full', graph='phys'))

# Market-level physical adjacency, pushed down from the city graph through the assignment
# matrix, because these baselines propagate over markets rather than over cities.
A_MKT = S @ A_phys @ S.T
np.fill_diagonal(A_MKT, 0.0)
A_MKT = A_MKT / np.maximum(A_MKT.sum(1, keepdims=True), 1e-8)
A_MKT_t = torch.tensor(A_MKT, dtype=torch.float32, device=DEVICE)
print('market-level A_phys:', A_MKT.shape, 'density %.3f' % (A_MKT > 0).mean())

def train_torch_baseline(factory, seed, epochs=None):
    """Same loaders, loss, horizon weights, optimiser and selection rule as the ladder, so
    a baseline never loses on budget rather than on merit."""
    torch.manual_seed(seed); np.random.seed(seed)
    epochs = epochs or SCFG['epochs']
    model = factory().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    dtr = make_loader_s(train_o, True,  BASE_XN, BASE_XG)
    dva = make_loader_s(val_o,   False, BASE_XN, BASE_XG)
    dte = make_loader_s(TEST_ORIGINS, False, BASE_XN, BASE_XG)
    W = HW_t.view(1, 1, -1)

    def step(b):
        xnb, xgb, y, ob, spk, anc = [z.to(DEVICE) for z in b]
        p = model(xnb, xgb) + anc.unsqueeze(-1)      # same persistence anchor as the ladder
        return p, y, ob

    def val():
        model.eval(); n = d = 0.0
        with torch.no_grad():
            for b in dva:
                p, y, ob = step(b)
                n += ((p - y).abs() * ob * W).sum().item(); d += (ob * W).sum().item()
        return n / max(d, 1e-9)

    best, bad, bstate = np.inf, 0, None
    for ep in range(epochs):
        model.train()
        for b in dtr:
            p, y, ob = step(b)
            loss = ((p - y).abs() * ob * W).sum() / (ob * W).sum().clamp(min=1e-9)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()
        v = val()
        if v < best - 1e-6:
            best, bad = v, 0
            bstate = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= (SCFG['patience'] if epochs > 5 else 2): break
    if bstate: model.load_state_dict(bstate)
    model.eval(); out = []
    with torch.no_grad():
        for b in dte:
            p, _, _ = step(b)
            out.append(p.cpu().numpy())
    P = np.concatenate(out, 0)
    assert P.shape == (len(TEST_ORIGINS), Y.shape[1], CFG['H']), P.shape
    return P

Fn_b, d_, H_ = BASE_XN.shape[-1], CFG['d'], CFG['H']
NEURAL_FAMILIES = {
    'LSTM':        ('temporal',  lambda: _Seq('lstm', Fn_b, d_, H_)),
    'GRU':         ('temporal',  lambda: _Seq('gru',  Fn_b, d_, H_)),
    'TCN':         ('temporal',  lambda: _Seq('tcn',  Fn_b, d_, H_)),
    'NHiTS':       ('temporal',  lambda: _NHiTS(Fn_b, d_, H_)),
    'STGCN_phys':  ('fixed',     lambda: _STGCN(Fn_b, d_, H_, A_MKT_t)),
    'AGCRN_style': ('adaptive',  lambda: _AdaptiveGraph(Fn_b, d_, H_, Y.shape[1], mode='gru')),
    'GWNet_style': ('adaptive',  lambda: _AdaptiveGraph(Fn_b, d_, H_, Y.shape[1], mode='tcn')),
}
for _n, (_fam, _f) in NEURAL_FAMILIES.items():
    # no 'fn': the runner dispatches on the presence of 'factory'
    BASELINES[_n] = dict(family=_fam, factory=_f)

print('registered %d baselines total' % len(BASELINES))
for _f in ('naive', 'statistical', 'ml', 'temporal', 'fixed', 'adaptive'):
    print('   %-12s %s' % (_f, [k for k, v in BASELINES.items() if v['family'] == _f]))
print('')
print('Dynamic-graph family: L4_price_dynamic in the ladder is the comparator,')
print('protocol allows ("or a feasible contemporary comparator").')

In [ ]:
# === P5. Metric suite (protocol 5.5) ===
# MAE and RMSE in IDR, MASE and skill against persistence, and the distributional views the
# protocol asks for: per-market spread, the worst market decile, high-volatility windows,
# and price-spike days. Aggregate MAE alone hides exactly the cases a food-price model is
# bought for.
# The protocol reports MAE in IDR. Y is per-market z-scored log price, so a metric computed
# on it is in z-units and is NOT comparable to v4's Table 6 -- the smoke run made this
# obvious by printing a whole table of 0.0 and 1.0. This is the numpy twin of v4's
# to_rp(z) = exp(z*sd + mu), applied to predictions and targets before any error is taken.
def RP(z):
    return np.exp(z * sd[None, :, None] + mu[None, :, None])

def metric_block(pred, te, tag=''):
    """pred: (T_te, N, H) in model units. Returns a dict of every protocol metric."""
    yt = RP(np.stack([Y[t + 1:t + 1 + CFG['H']].T for t in te]))    # (T,N,H) in IDR
    ob = np.stack([obs[t + 1:t + 1 + CFG['H']].T for t in te])
    per = RP(np.stack([np.repeat(Y[t][:, None], CFG['H'], 1) for t in te]))
    pred = RP(pred)
    out = {}
    for hi, h in enumerate(CFG['HORIZONS']):
        k = h - 1
        e  = np.abs(pred[:, :, k] - yt[:, :, k]) * ob[:, :, k]
        ep = np.abs(per[:, :, k]  - yt[:, :, k]) * ob[:, :, k]
        n  = np.maximum(ob[:, :, k].sum(), 1)
        mae  = e.sum() / n
        rmse = np.sqrt((((pred[:, :, k] - yt[:, :, k]) ** 2) * ob[:, :, k]).sum() / n)
        mase = mae / max(ep.sum() / n, 1e-12)
        # per-market MAE, for the spread and worst-decile views
        pm = e.sum(0) / np.maximum(ob[:, :, k].sum(0), 1)
        # high-volatility windows: top quintile of realised |dlog| over the target span
        vol = np.abs(np.diff(yt[:, :, :k + 1], axis=2)).mean((1, 2)) if k > 0 else np.zeros(len(te))
        hv = vol >= np.quantile(vol, 0.8) if k > 0 else np.zeros(len(te), bool)
        # spike defined on the log scale, as in v4, not on the Rupiah difference
        spike = np.abs(np.log(np.maximum(yt[:, :, k], 1e-9))
                       - np.log(np.maximum(per[:, :, k], 1e-9))) >= SCFG['spike_threshold']
        out[h] = dict(
            MAE=float(mae), RMSE=float(rmse), MASE=float(mase),
            skill_vs_persistence=float(1 - mae / max(ep.sum() / n, 1e-12)),
            market_MAE_sd=float(pm.std()),
            worst_decile_MAE=float(np.quantile(pm, 0.9)),
            highvol_MAE=float((e[hv].sum() / max(ob[:, :, k][hv].sum(), 1)) if hv.any() else np.nan),
            spike_MAE=float((e * spike).sum() / max((ob[:, :, k] * spike).sum(), 1)),
            spike_share=float(spike.mean()))
    return out

def metrics_frame(store):
    rows = []
    for (name, seed), pred in store.items():
        for h, m in metric_block(pred, TEST_ORIGINS).items():
            rows.append(dict(model=name, seed=seed, h=h, **m))
    return pd.DataFrame(rows)

print('metrics: MAE, RMSE, MASE, skill, market spread, worst decile, high-volatility, spikes')

In [ ]:
# === H1. The hierarchy as an explicit switch ===
# The ladder in P1 varies the adjacency. This varies the market->city ASSIGNMENT, which no
# notebook in the corpus has ever done. Everything else -- encoder, context encoder, fusion
# gate, head, loss, budget, selection rule -- is untouched.
HSMOKE = False         # False: the real run (10 seeds, 50 epochs). True: 1 seed, 3 epochs.

HSEEDS = 1 if HSMOKE else SCFG['seeds']
HEPOCHS = 3 if HSMOKE else SCFG['epochs']
HIER_HEADS = ('R', 'S')          # both, for the same reason the ladder runs both
print('hierarchy run: %d seeds x %d epochs x %d heads' % (HSEEDS, HEPOCHS, len(HIER_HEADS)))

# --- a 1-D ordering of cities, used only to build the coarse grouping ------------------
# Classical MDS on the physical distance matrix, first coordinate. Deterministic, numpy
# only, and it puts geographically adjacent cities next to each other, which is what the
# coarse rung needs. It is never used by any model.
def _mds1(D):
    D2 = D ** 2
    J = np.eye(D.shape[0]) - np.ones_like(D2) / D.shape[0]
    B = -0.5 * J @ D2 @ J
    w, V = np.linalg.eigh(B)
    return V[:, -1] * np.sqrt(max(w[-1], 1e-12))

# The distance matrix, rebuilt from source rather than reused by name. Cell 2 binds it to
# `M`, but P7 rebinds `M` to metrics_frame(STORE) -- a DataFrame -- so by the time this cell
# runs the array is gone and `M[np.ix_(perm, perm)]` indexes a DataFrame by tuple. Re-reading
# is two lines and cannot be shadowed by anything that runs in between.
try:
    _Dm = pd.read_excel('Matrix_Jarak_Jawa_Timur_simetris.xlsx')
    DIST_CITY = _Dm.iloc[:, 1:].to_numpy(float)[np.ix_(perm, perm)]
    assert DIST_CITY.shape == (R, R), 'distance matrix is %s, expected (%d, %d)' % (
        DIST_CITY.shape, R, R)
    assert np.allclose(np.diag(DIST_CITY), 0), 'distance matrix has a non-zero diagonal'
    _city_order = np.argsort(_mds1(DIST_CITY))
except Exception as _e:
    # _city_order feeds only the 'coarse' rung, so a failure here must not take the cell down.
    print('WARNING: could not rebuild the distance matrix (%s).' % _e)
    print('         falling back to index order; the coarse rung becomes an arbitrary')
    print('         grouping and should not be read as a geographic one.')
    DIST_CITY, _city_order = None, np.arange(R)

# `perm` maps 38 cities onto the distance workbook's rows through a normalised name, and East
# Java has seven Kabupaten/Kota pairs sharing a name. Those pairs therefore share a row and
# sit at distance ZERO from each other in A_phys -- defensible, since they are the same
# location, but the paper should say so rather than leave it implicit in Section 3.3.
_dup = R - len(set(perm.tolist()))
_rs = A_phys.sum(1)
if _dup or not np.allclose(_rs, 1.0, atol=1e-3):
    print('')
    print('  !! PHYSICAL GRAPH INTEGRITY -- pre-existing, not introduced here')
    print('  cities sharing a distance-matrix row : %d of %d' % (_dup, R))
    print('  A_phys row sums                      : mean %.4f, min %.4f' % (_rs.mean(), _rs.min()))
    print('  Section 3.3 says the physical graph is row-normalised. It is not, and the two')
    print('  facts have one cause. _norm() in cell 2 strips both "Kabupaten " and "Kota ",')
    print('  so the workbook rows "Blitar Kab." and "Kota Blitar" collapse to one key; the')
    print('  lookup keeps the last, every Kabupaten inherits its Kota twin\'s distances, and')
    print('  seven genuine rows are never used. Row normalisation runs BEFORE that reindex,')
    print('  so the reindexed rows no longer sum to one.')
    print('  Consequence: each collapsed pair sits at distance 0, hence kernel weight 1.0,')
    print('  hence guaranteed to be the other\'s strongest physical neighbour. Malang and')
    print('  Kediri are two of the seven -- the cities on the documented chilli chain that')
    print('  section 7.2 is supposed to validate against.')
    print('  L8 puts the anchor\'s accuracy contribution below 0.4%, so this is a validity')
    print('  and description problem rather than an accuracy one. Fix it BEFORE the long')
    print('  run, not after: see the note at the end of this notebook.')
    print('')

def _row_norm(A):
    A = np.asarray(A, np.float32).copy()
    np.fill_diagonal(A, 0.0)
    return (A / np.maximum(A.sum(1, keepdims=True), 1e-8)).astype(np.float32)

# Market-level physical graph, pushed down through the true assignment. Same construction
# as P4's A_MKT -- repeated here so this cell does not depend on P4 having been run.
A_MKT_H = _row_norm(S @ A_phys @ S.T)

_true_lab = np.array([int(np.argmax(S[j])) for j in range(N)])
assert (np.bincount(_true_lab, minlength=R) == S.sum(0)).all()


def hier_spec(kind, seed):
    """Return (S_new, Sbar_new, Aphys_new, R_new, graph_mode_override, note).

    seed enters only for 'perm', where it draws the random assignment. Using the run seed
    means the ten seeds explore ten different permutations, so the null is a distribution
    over assignments as well as over initialisations rather than one arbitrary draw.
    """
    if kind == 'real':
        return S, Sbar, A_phys, R, None, 'true market-city assignment'

    if kind == 'perm':
        lab = _true_lab.copy()
        np.random.default_rng(9000 + seed).shuffle(lab)      # permutes labels across nodes
        Sp = np.zeros((N, R), np.float32)
        Sp[np.arange(N), lab] = 1.0
        # City sizes are preserved exactly, so pooling denominators and the physical graph
        # are unchanged. Only the membership is wrong.
        assert (Sp.sum(0) == S.sum(0)).all(), 'permutation changed city sizes'
        return Sp, Sp / Sp.sum(0, keepdims=True), A_phys, R, None, 'markets shuffled across cities'

    if kind == 'flat':
        I = np.eye(N, dtype=np.float32)
        return I, I, A_MKT_H, N, None, 'S = I: pooling is an identity, graph is market-level'

    if kind == 'single':
        one = np.ones((N, 1), np.float32)
        return one, one / N, np.ones((1, 1), np.float32), 1, None, 'all markets in one city'

    if kind == 'coarse':
        K = 10
        grp = np.zeros(R, int)
        for g, chunk in enumerate(np.array_split(_city_order, K)):
            grp[chunk] = g
        Mg = np.zeros((R, K), np.float32)          # city -> region, not the distance matrix
        Mg[np.arange(R), grp] = 1.0
        Sc = (S @ Mg).astype(np.float32)
        Ac = _row_norm(Mg.T @ A_phys @ Mg)
        return Sc, Sc / Sc.sum(0, keepdims=True), Ac, K, None, '38 cities collapsed to 10 regions'

    if kind in ('phys_price', 'static_price', 'uniform_price'):
        # The decisive control the full run left open. L2 (fixed physical) and L3 (learned
        # static) beat L1 by 5.6% and 7.7% at h=7, so message passing helps -- but nothing
        # yet says whether that is because the graph encodes SPATIAL STRUCTURE or merely
        # because averaging over neighbours is shrinkage. A uniform row-stochastic graph
        # pools exactly as much and encodes nothing. Run all three through one code path at
        # price-only context so the only difference is the adjacency.
        gm = {'phys_price': 'phys', 'static_price': 'static', 'uniform_price': 'phys'}[kind]
        Ax = _row_norm(np.ones((R, R), np.float32)) if kind == 'uniform_price' else A_phys
        return S, Sbar, Ax, R, gm, 'price-only context, %s adjacency' % gm

    if kind in ('shortonly', 'longonly'):
        # Identical model and identical hierarchy. Only the horizon weighting in the loss
        # differs, applied in train_hier_rung.
        return S, Sbar, A_phys, R, None, 'real hierarchy, loss scored on one horizon band'

    if kind in ('freegate', 'freeboth'):
        # Same hierarchy as 'real'. What differs is the PARAMETERISATION, applied in
        # build_hier_model: these rungs are allowed to reach the graph-free configuration.
        return S, Sbar, A_phys, R, None, 'real hierarchy, floors removed'

    if kind == 'uniform':
        # Protocol 7.4's null graph, TRAINED rather than swapped in at inference. graph_mode
        # is forced to 'phys' so the uniform matrix IS the adjacency: every city receives the
        # same message from every other. If a rung with no edge structure matches the learned
        # one, no learned edge is carrying information.
        U = _row_norm(np.ones((R, R), np.float32))
        return S, Sbar, U, R, 'phys', 'uniform row-stochastic null graph'

    raise KeyError(kind)


# The constructive experiment. Section 3 argues the graph-free model is unreachable:
#   Actx is row-stochastic, alpha in [0.2, 0.8], gate in [0.15, 0.85], so at least 15% of
#   every market's state is the city message no matter what the optimiser wants.
# These two rungs remove the floors and ask whether the model FINDS the null when allowed
# to. That is a positive experiment, and it is the one the paper's Section 3 promises.
ARCH = {
    'freegate': dict(gate_min=0.0, gate_span=1.0),                       # gate may reach 1
    'freeboth': dict(gate_min=0.0, gate_span=1.0,
                     alpha_min=0.0, alpha_span=1.0),                     # anchor may vanish too
}

# Section 3's prediction, made testable without touching the architecture.
#
# The claim is that the fusion gate sits at ~0.6 and refuses to move because ONE gate must
# serve thirty horizons: the city message is a common-factor estimator, which is information
# at h=30 (PC1 = 0.76) and contamination at h=1 (PC1 = 0.12), and the horizon-weighted loss
# forces a compromise. If that is right, removing the conflict should move the gate --- up
# when only short horizons are scored, down when only long ones are.
#
# This needs no new modules and no change to forward(): it is the same model trained under a
# different horizon weighting. That makes it the safest possible test of the claim, and a
# sharp one, because the two rungs are predicted to move in OPPOSITE directions.
HW_OVERRIDE = {
    'shortonly': np.array([1.0 if h <= 3 else 0.0 for h in range(1, CFG['H'] + 1)], np.float32),
    'longonly':  np.array([1.0 if h >= 14 else 0.0 for h in range(1, CFG['H'] + 1)], np.float32),
}

# Rungs whose context block is price-only rather than the full context of L6. Applied in
# build_hier_model. Everything else about them is identical to the other rungs.
CTX_OVERRIDE = {'phys_price': 'price', 'static_price': 'price', 'uniform_price': 'price'}

# Heads to run per rung. shortonly/longonly are gate diagnostics, and the R head is the one
# Table 9 assigns to h<=14, so running them under S as well doubles the cost for nothing.
HEADS_FOR = {'shortonly': ('R',), 'longonly': ('R',)}

HIER = ['real', 'perm', 'flat', 'single', 'coarse', 'uniform', 'freegate', 'freeboth',
        'shortonly', 'longonly', 'phys_price', 'static_price', 'uniform_price']
HIER_CTX = 'L6_ctx_conditioned'    # the hierarchy is tested inside the proposed model

# A control that silently equals the thing it controls is worse than no control. P6 learned
# this the hard way with L7; check it here before spending GPU time.
_s0 = hier_spec('perm', 0)[0]
assert not np.allclose(_s0, S), 'perm rung inert: its assignment equals the real one'
_agree = float((np.argmax(_s0, 1) == _true_lab).mean())
print('perm check: %.1f%% of markets keep their true city by chance (expect ~%.1f%%)'
      % (100 * _agree, 100.0 / R))

print('')
for k in HIER:
    Sx, Sbx, Ax, Rx, gm, note = hier_spec(k, 0)
    print('  %-8s R=%-4d A %-9s %s' % (k, Rx, str(Ax.shape), note))

In [ ]:
# === H2. Binding the hierarchy switch to the training machinery ===
# train_ladder_rung hardcodes Sbar_t / S_t / Aphys_t and builds CMILModes at the global R.
# This is the same function with those four quantities lifted into arguments. Nothing else
# differs, so a hierarchy rung and a ladder rung are trained by identical code.
def hier_name(kind, head):
    return 'H_%s@%s' % (kind, head)


def build_hier_model(kind, seed, head='R'):
    Sx, Sbx, Ax, Rx, gm, _ = hier_spec(kind, seed)
    cfg = dict(LADDER[HIER_CTX])
    if kind in CTX_OVERRIDE:
        cfg['ctx'] = CTX_OVERRIDE[kind]
    if gm is not None:
        cfg['graph'] = gm
    xn, xg, ns, gs, dom = ladder_arrays(cfg)
    sc = dict(LADDER_SC); sc.update(HEADS[head])
    m = CMINScenario(xn.shape[-1], xg.shape[-1], CFG['d'], CFG['d_ctx'], CFG['H'],
                     ns, gs, dom, sc)
    m.cmil = CMILModes(CFG['d'], CFG['d_ctx'], Rx, xg.shape[-1],
                       n_heads=CFG.get('n_heads', 4),
                       context_alpha=CFG.get('context_alpha', True),
                       graph_mode=cfg['graph'])
    if not cfg.get('anchor', True):
        m.cmil.alpha_min, m.cmil.alpha_span = 0.0, 0.0
    ov = ARCH.get(kind, {})
    if 'gate_min' in ov:
        m.gate_min, m.gate_span = ov['gate_min'], ov['gate_span']
    if 'alpha_min' in ov:
        m.cmil.alpha_min, m.cmil.alpha_span = ov['alpha_min'], ov['alpha_span']
    t = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=torch.float32, device=DEVICE)
    return m.to(DEVICE), (xn, xg), (t(Sbx), t(Sx), t(Ax))


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def train_hier_rung(kind, seed, head='R', epochs=None, name=None):
    torch.manual_seed(seed); np.random.seed(seed)
    epochs = epochs or HEPOCHS
    model, (xn, xg), (Sb_t, Sx_t, Ax_t) = build_hier_model(kind, seed, head)
    opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'],
                           weight_decay=CFG['weight_decay'])
    dtr = make_loader_s(train_o, True,  xn, xg)
    dva = make_loader_s(val_o,   False, xn, xg)
    dte = make_loader_s(TEST_ORIGINS, False, xn, xg)
    # Horizon weighting: the stock vector unless this rung deliberately scores one band.
    _hw = HW_OVERRIDE.get(kind)
    W = (torch.tensor(_hw, device=DEVICE) if _hw is not None else HW_t).view(1, 1, -1)

    def fwd(b):
        xnb, xgb, y, ob, spk, anc = [z.to(DEVICE) for z in b]
        out = model(xnb, xgb, Sb_t, Sx_t, Ax_t)
        p = out[0] if isinstance(out, (tuple, list)) else out
        return p, y, ob

    def score(dl):
        model.eval(); num = den = 0.0
        with torch.no_grad():
            for b in dl:
                p, y, ob = fwd(b)
                num += ((p - y).abs() * ob * W).sum().item()
                den += (ob * W).sum().item()
        return num / max(den, 1e-9)

    best, bad, bstate = np.inf, 0, None
    for ep in range(epochs):
        model.train()
        for b in dtr:
            p, y, ob = fwd(b)
            loss = ((p - y).abs() * ob * W).sum() / ((ob * W).sum().clamp(min=1e-9))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip']); opt.step()
        v = score(dva)
        if v < best - 1e-6:
            best, bad = v, 0
            bstate = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= (SCFG['patience'] if epochs > 5 else 2):
                break
    if bstate:
        model.load_state_dict(bstate)
        if name:
            torch.save(bstate, rung_ckpt(name, seed))

    preds = []
    model.eval()
    with torch.no_grad():
        for b in dte:
            p, _, _ = fwd(b)
            preds.append(p.cpu().numpy())
    P = np.concatenate(preds, 0)
    assert P.shape == (len(TEST_ORIGINS), Y.shape[1], CFG['H']), P.shape
    return P


def load_hier_weights(model, name, seed, quiet=False):
    """Load saved weights, and say so out loud when it cannot.

    This deliberately REDEFINES the version from P2, which returned a bare False. That is
    how a whole diagnostics section came back empty once: the False became a None inside
    graph_series, every caller skipped, and the run printed section headers with nothing
    under them and no error. On a Drive mount a missing checkpoint is usually latency
    rather than absence -- files written in an earlier session may not be visible yet -- so
    retry briefly, then report which case it was.
    """
    f = rung_ckpt(name, seed)
    for attempt in range(3):
        if os.path.exists(f):
            model.load_state_dict(torch.load(f, map_location=DEVICE))
            return True
        if attempt < 2:
            time.sleep(2.0)
            try:
                os.listdir(os.path.dirname(f))   # force a FUSE directory refresh
            except OSError:
                pass
    if not quiet:
        print('    [missing checkpoint] %s' % os.path.basename(f))
    return False


# Every later diagnostic resolves this name at call time, so the override reaches them all.
load_rung_weights = load_hier_weights


print('=== parameter count by rung (head S) ===')
for k in HIER:
    m, _, _ = build_hier_model(k, 0, 'S')
    print('  %-8s %8d params' % (k, n_params(m)))
    del m
print('')
print('real and perm MUST match exactly -- that is what makes perm the decisive control.')
print('flat/single/coarse differ through the city embedding (R x d) and are read with that')
print('in mind; they answer a granularity question, not the geography question.')

## F0-F12 -- the final run

In [ ]:
# === F0. The queue, the variant rungs, and two guards that must pass before training ===
#
# Nothing in this notebook deletes, moves or overwrites an existing file.  Training writes a
# checkpoint only where none exists; every analysis writes into protocol/final/.
import os, glob, re, time, json, itertools, zipfile, contextlib
from math import comb
import numpy as np, pandas as pd, torch

DRY_RUN = globals().get('DRY_RUN', False)
N_SEEDS = 10

# ---- original checkpoints ----------------------------------------------------------------
# The ten-seed checkpoints behind the published tables were found in the 1 September protocol
# export.  Upload cmin_original_checkpoints.zip into the Drive folder cmin_v4_out/ and this cell
# unpacks it into a NEW folder, protocol_original/, and reads and writes there.  Nothing in the
# existing protocol/ folder is touched, and a file already unpacked is never overwritten.  With
# the originals in place, tier T1 needs no training and F3 compares like with like.
ORIG_ZIP = os.path.join(OUT_DIR, 'cmin_original_checkpoints.zip')
ORIG_DIR = os.path.join(OUT_DIR, 'protocol_original')
if os.path.exists(ORIG_ZIP) or os.path.isdir(ORIG_DIR):
    os.makedirs(ORIG_DIR, exist_ok=True)
    if os.path.exists(ORIG_ZIP):
        with zipfile.ZipFile(ORIG_ZIP) as _z:
            _new = 0
            for _n in _z.namelist():
                _dst = os.path.join(ORIG_DIR, os.path.basename(_n))
                if _n.endswith('.pt') and not os.path.exists(_dst):
                    with open(_dst, 'wb') as _fh:
                        _fh.write(_z.read(_n))
                    _new += 1
        print('original checkpoints: %d unpacked into %s' % (_new, ORIG_DIR))
    PROTO_DIR = ORIG_DIR
else:
    print('original checkpoints not found at %s -- using %s' % (ORIG_ZIP, PROTO_DIR))
assert SCFG['seeds'] == N_SEEDS and SCFG['epochs'] == 50, \
    'expected the ten-seed, fifty-epoch protocol; got %s seeds, %s epochs' % (SCFG['seeds'], SCFG['epochs'])
FIN_DIR = os.path.join(PROTO_DIR, 'final')
EXT_DIR = os.path.join(FIN_DIR, 'extract')
os.makedirs(EXT_DIR, exist_ok=True)
print('checkpoints :', PROTO_DIR)
print('results     :', FIN_DIR)
print('device      :', DEVICE)

# ---- one context-conditioning point removed at a time ------------------------------------
# Each variant is the full model L6 with the context INPUT of one submodule replaced by
# zeros.  No parameter is added or removed, so the parameter count equals L6's and a variant
# at seed s starts from exactly L6's initial weights at seed s.  The removed point becomes a
# learned constant: FiLM a fixed affine map, the head mixture a fixed softmax, the
# temperature a fixed scalar, the anchor a fixed weight.  That is the ladder's own
# convention -- zero an input rather than delete a module -- applied to each point in turn.
def _zero_input(mod):
    f = mod.forward
    mod.forward = lambda x, _f=f: _f(torch.zeros_like(x))

PATCHES = {
    'film':  lambda m: (_zero_input(m.cmil.gamma), _zero_input(m.cmil.beta)),
    'heads': lambda m: _zero_input(m.cmil.head_gate),
    'temp':  lambda m: _zero_input(m.cmil.context_edge_bias),
    'alpha': lambda m: _zero_input(m.cmil.alpha_head),
}
VARIANTS = {}
for _k in PATCHES:
    VARIANTS['L6no%s@R' % _k] = dict(base='L6_ctx_conditioned', head='R', cfg={}, patch=_k)
for _k in (1, 2, 8):
    VARIANTS['L6nh%d@R' % _k] = dict(base='L6_ctx_conditioned', head='R', cfg={'n_heads': _k}, patch=None)
for _k in (32, 128):
    VARIANTS['L6d%d@R' % _k] = dict(base='L6_ctx_conditioned', head='R', cfg={'d': _k}, patch=None)
# L6 itself, retrained in THIS session.  The variants are trained here, the original L6 was
# trained on 28 August on whatever GPU Colab assigned then; comparing across sessions would mix
# the mechanism with the environment.  This replicate is the reference F11 uses.
VARIANTS['L6same@R'] = dict(base='L6_ctx_conditioned', head='R', cfg={}, patch=None)


@contextlib.contextmanager
def cfg_override(**kw):
    old = {k: (k in CFG, CFG.get(k)) for k in kw}
    CFG.update(kw)
    try:
        yield
    finally:
        for k, (had, v) in old.items():
            if had:
                CFG[k] = v
            else:
                CFG.pop(k, None)


if '_BASE_BUILD_LADDER' not in globals():        # capture once; a re-run must not wrap a wrapper
    _BASE_BUILD_LADDER = build_ladder_model


def build_variant(name):
    v = VARIANTS[name]
    with cfg_override(**v['cfg']):
        m, arrs = _BASE_BUILD_LADDER(LADDER[v['base']], v['head'])
    if v['patch']:
        PATCHES[v['patch']](m)
    return m, arrs


def train_variant(name, seed):
    """Train a variant through the UNCHANGED train_ladder_rung, so its optimiser, loaders,
    early stopping and checkpointing are byte-for-byte those of every other rung."""
    g = globals()
    g['build_ladder_model'] = lambda cfg, head='R', _n=name: build_variant(_n)
    try:
        return train_ladder_rung(LADDER[VARIANTS[name]['base']], seed, epochs=SCFG['epochs'],
                                 name=name, head=VARIANTS[name]['head'])
    finally:
        g['build_ladder_model'] = _BASE_BUILD_LADDER


def build_any(name, seed=0):
    """(model, (xn, xg), (Sbar, S, Aphys)) for any rung name used in this notebook."""
    if name in VARIANTS:
        m, arrs = build_variant(name)
        return m, arrs, (Sbar_t, S_t, Aphys_t)
    base, hd = name.split('@')
    if base.startswith('H_'):
        return build_hier_model(base[2:], seed, hd)
    m, arrs = _BASE_BUILD_LADDER(LADDER[base], hd)
    return m, arrs, (Sbar_t, S_t, Aphys_t)


def capacity_and_patch_checks():
    """Two guards.  A variant with different capacity would confound the contrast, and a
    patch that silently did nothing would make its control identical to the thing it
    controls -- the L7 bug this protocol has already paid for once."""
    ref, (xn, xg), (Sb, Sx, Ax) = build_any('L6_ctx_conditioned@R')
    n_ref = sum(p.numel() for p in ref.parameters())
    b = next(iter(make_loader_s(list(TEST_ORIGINS[:16]), False, xn, xg)))
    xnb, xgb = b[0].to(DEVICE), b[1].to(DEVICE)
    ref.eval()
    with torch.no_grad():
        _, _, aux_ref = ref(xnb, xgb, Sb, Sx, Ax)
    print('\n=== guard 1: capacity.  guard 2: the removed point is really constant ===')
    print('  %-22s %8d params' % ('L6_ctx_conditioned@R', n_ref))
    KEY = {'heads': 'head_w', 'temp': 'temperature', 'alpha': 'alpha'}
    for nm, v in VARIANTS.items():
        m, _, _ = build_any(nm)
        n = sum(p.numel() for p in m.parameters())
        note = ''
        if v['patch']:
            assert n == n_ref, '%s has %d params, L6 has %d' % (nm, n, n_ref)
            m.load_state_dict(ref.state_dict())
            m.eval()
            with torch.no_grad():
                _, _, aux = m(xnb, xgb, Sb, Sx, Ax)
            k = KEY.get(v['patch'])
            if k:
                before = float(aux_ref[k].float().std(0).max())
                after = float(aux[k].float().std(0).max())
                assert before > 1e-6, 'guard vacuous: %s does not vary even in L6' % k
                assert after < 1e-6, '%s: %s still varies across windows (%.2e)' % (nm, k, after)
                note = '%s spread across windows %.1e -> %.1e' % (k, before, after)
            else:
                g_ref = ref.cmil.gamma(torch.randn(4, CFG['d_ctx'], device=DEVICE))
                g_new = m.cmil.gamma(torch.randn(4, CFG['d_ctx'], device=DEVICE))
                assert float(g_ref.std(0).max()) > 1e-6 and float(g_new.std(0).max()) < 1e-6
                note = 'FiLM gamma/beta constant in context'
        if not note:
            note = 'replicate of L6 in this session' if nm == 'L6same@R' else 'sensitivity rung (capacity differs by design)'
        print('  %-22s %8d params  %s' % (nm, n, note))
        del m
    del ref
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()


# ---- the queue ---------------------------------------------------------------------------
TIERS = {
    1: [('ladder', 'L1_encoder_only', 'R'), ('ladder', 'L3_learned_static', 'R'),
        ('ladder', 'L6_ctx_conditioned', 'R'), ('hier', 'uniform_price', 'R'),
        ('ladder', 'L7_ctx_shuffled', 'S')],
    2: [('variant', 'L6same@R', None)] + [('variant', 'L6no%s@R' % k, None) for k in PATCHES],
    3: [('variant', n, None) for n in VARIANTS if n.startswith(('L6nh', 'L6d'))],
}
RUN_TIERS = (1, 2, 3)          # drop a tier here to skip it


def qname(kind, a, head):
    return rung_name(a, head) if kind == 'ladder' else (hier_name(a, head) if kind == 'hier' else a)


QUEUE = []
for _t in RUN_TIERS:
    for _s in range(N_SEEDS):  # seed-major: an interrupted tier still has balanced seeds
        for kind, a, head in TIERS[_t]:
            QUEUE.append((_t, kind, a, head, _s, qname(kind, a, head)))

MIN_PER_RUN = 1.15
print('\n=== queue ===')
print('%-4s %-24s %6s %8s' % ('tier', 'rung', 'have', 'to train'))
for _t in RUN_TIERS:
    for nm in dict.fromkeys(q[5] for q in QUEUE if q[0] == _t):
        h = sum(os.path.exists(rung_ckpt(nm, s)) for s in range(N_SEEDS))
        print('T%-3d %-24s %6d %8d' % (_t, nm, h, N_SEEDS - h))
_todo = [q for q in QUEUE if not os.path.exists(rung_ckpt(q[5], q[4]))]
print('\n%d runs to train, roughly %.0f minutes on a T4' % (len(_todo), len(_todo) * MIN_PER_RUN))
_hr = sum(os.path.exists(rung_ckpt('H_real@S', s)) for s in range(N_SEEDS))
print('H_real@S checkpoints on Drive: %d  (F7 uses them to trace where r = 0.987 came from)' % _hr)

if not DRY_RUN:
    capacity_and_patch_checks()

In [ ]:
# === F1. Train everything missing.  Resumable: after a disconnect, run the notebook again. ===
if DEVICE != 'cuda':
    raise RuntimeError('F1 trains up to %d runs and needs a GPU; this session is on %s. '
                       'Runtime -> Change runtime type -> T4 GPU.' % (len(_todo), DEVICE))
t0 = time.time()
todo = [q for q in QUEUE if not os.path.exists(rung_ckpt(q[5], q[4]))]
n_done, failed, bad = 0, {}, set()
print('%d runs to train' % len(todo))
for i, (tier, kind, a, head, seed, nm) in enumerate(todo):
    if nm in bad or os.path.exists(rung_ckpt(nm, seed)):
        continue
    try:
        if kind == 'ladder':
            train_ladder_rung(LADDER[a], seed, epochs=SCFG['epochs'], name=nm, head=head)
        elif kind == 'hier':
            train_hier_rung(a, seed, head=head, epochs=HEPOCHS, name=nm)
        else:
            train_variant(nm, seed)
        n_done += 1
        el = time.time() - t0
        print('  T%d %-22s s%d  %s  %.1f min elapsed, ~%.0f min left'
              % (tier, nm, seed, 'saved' if os.path.exists(rung_ckpt(nm, seed)) else 'NO CHECKPOINT',
                 el / 60, el / n_done * (len(todo) - i - 1) / 60))
    except Exception as e:
        failed[(nm, seed)] = '%s: %s' % (type(e).__name__, e)
        bad.add(nm)
        print("  T%d %-22s s%d  FAILED -> %s  (this rung's remaining seeds are skipped)"
              % (tier, nm, seed, failed[(nm, seed)]))
    finally:
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
print('\ntrained %d runs in %.1f min' % (n_done, (time.time() - t0) / 60))
print('failed:', failed if failed else 'none')

In [ ]:
# === F2a. Definitions shared by every analysis (no model is touched here) ===
EXT_VERSION = 'final-v1'
ALL_ORIGINS = list(train_o) + list(val_o) + list(TEST_ORIGINS)
N_TV = len(train_o) + len(val_o)
ORIG_DATE = pd.DatetimeIndex([dates[t] for t in ALL_ORIGINS])
IS_TEST = np.arange(len(ALL_ORIGINS)) >= N_TV
_YT = RP(np.stack([Y[t + 1:t + 1 + CFG['H']].T for t in TEST_ORIGINS]))
_OB = np.stack([obs[t + 1:t + 1 + CFG['H']].T for t in TEST_ORIGINS])
HZ = list(CFG['HORIZONS'])
# RP() converts model units to rupiah through the GLOBALS mu and sd from cell 4.  A top-level
# loop written `for sd in seeds` silently rebinds sd to an integer, and every MAE after it is
# wrong without an error.  Snapshot both and check them wherever a price is computed.
_MU0, _SD0 = np.array(mu, copy=True), np.array(sd, copy=True)


def _scale_intact():
    ok_sd = isinstance(sd, np.ndarray) and sd.shape == _SD0.shape and np.allclose(sd, _SD0)
    ok_mu = isinstance(mu, np.ndarray) and mu.shape == _MU0.shape and np.allclose(mu, _MU0)
    assert ok_sd, 'global sd was overwritten -- a loop variable is shadowing the price scale'
    assert ok_mu, 'global mu was overwritten'


def score_pred(P):
    """Per-window MAE at each horizon (for the paired bootstrap) and the observation-weighted
    MAE exactly as P5's metric_block computes it (for the reproduction check)."""
    _scale_intact()
    pred = RP(P)
    wm, mae = [], []
    for h in HZ:
        k = h - 1
        e = np.abs(pred[:, :, k] - _YT[:, :, k]) * _OB[:, :, k]
        wm.append(e.sum(1) / np.maximum(_OB[:, :, k].sum(1), 1.0))
        mae.append(e.sum() / max(_OB[:, :, k].sum(), 1.0))
    return np.stack(wm).astype(np.float32), np.asarray(mae, np.float64)


EXP1_ARMS = [('L1_encoder_only@R', 'no graph'), ('H_uniform_price@R', 'uniform adjacency'),
             ('L3_learned_static@R', 'learned static (CMIN-S)'),
             ('L6_ctx_conditioned@R', 'context-conditioned (CMIN)')]
EXP2_RUNGS = ['L6_ctx_conditioned@S', 'L6_ctx_conditioned@R']
FULL = [a for a, _ in EXP1_ARMS] + ['L6_ctx_conditioned@S', 'L7_ctx_shuffled@S']
PER_ORIGIN_A = {'L6_ctx_conditioned@S', 'L6_ctx_conditioned@R', 'L7_ctx_shuffled@S'}
ACC = list(VARIANTS)
PROV = ['H_real@S']
UNTRAINED = 'UNTRAINED_L6@S'
UNTRAINED_SEEDS = list(range(100, 100 + N_SEEDS))

REG = {0: 'normal', 1: 'ramadan', 2: 'peak_rain', 3: 'harvest'}


def regime_labels(origins):
    lab = np.zeros(len(origins), int)
    lab[np.asarray([peak_rain_flag[t] for t in origins]) == 1] = 2
    lab[np.asarray([harvest_flag[t] for t in origins]) == 1] = 3
    lab[np.asarray([ramadan_flag[t] for t in origins]) == 1] = 1   # Ramadan wins ties
    return lab


def ext_path(nm, seed_):
    return os.path.join(EXT_DIR, '%s__s%d.npz' % (nm, seed_))


def ext_seeds(nm):
    out = []
    for f in glob.glob(os.path.join(EXT_DIR, glob.escape(nm) + '__s*.npz')):
        m = re.match(r'.*__s(\d+)\.npz$', f)
        if m:
            out.append(int(m.group(1)))
    return sorted(out)


def load_ext(nm, seed_):
    return np.load(ext_path(nm, seed_), allow_pickle=False)   # lazy: an array is read only when indexed


def ci95(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan
    from scipy import stats
    return float(stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))


def sign_test(dv):
    dv = np.asarray(dv, float)
    dv = dv[np.isfinite(dv) & (dv != 0)]
    if len(dv) < 2:
        return np.nan
    k = int(max((dv > 0).sum(), (dv < 0).sum()))
    return min(1.0, 2 * sum(comb(len(dv), i) for i in range(k, len(dv) + 1)) / 2 ** len(dv))


PUBLISHED = {   # ten-seed means printed in the paper (data/mae_by_model_horizon.csv)
    'H_uniform_price@R':    [2100, 4681, 7126, 10348, 16221],
    'L1_encoder_only@R':    [2100, 4756, 7663, 11122, 16620],
    'L3_learned_static@R':  [2090, 4669, 7074, 10174, 15890],
    'L6_ctx_conditioned@R': [2132, 4917, 8140, 12252, 15169],
    'L6_ctx_conditioned@S': [3529, 5402, 7910, 12018, 15297],
    'L7_ctx_shuffled@S':    [3996, 5528, 7794, 11355, 16414],
    'L6same@R':             [2132, 4917, 8140, 12252, 15169],   # must match L6@R if the session does not matter
}
print('F2a definitions ready: %d origins (%d test)' % (len(ALL_ORIGINS), len(TEST_ORIGINS)))

In [ ]:
# === F2b. Inference: one file per model-seed.  Resumable, and the only model-touching analysis. ===
def extract(nm, seed_, full, untrained=False):
    if untrained:
        torch.manual_seed(seed_); np.random.seed(seed_)
        model, (xn, xg), (Sb, Sx, Ax) = build_any('L6_ctx_conditioned@S', seed_)
    else:
        model, (xn, xg), (Sb, Sx, Ax) = build_any(nm, seed_)
        if not load_rung_weights(model, nm, seed_, quiet=True):
            return False
    model.eval()
    origins = ALL_ORIGINS if full else list(TEST_ORIGINS)
    n_tv = N_TV if full else 0
    buf = {}
    hs = [model.cmil.register_forward_hook(lambda m, i, o: buf.__setitem__('cmil', o)),
          model.gate.register_forward_hook(lambda m, i, o: buf.__setitem__('g', o))]
    if full:
        hs += [model.node_attn.register_forward_hook(lambda m, i, o: buf.__setitem__('natt', o[1])),
               model.ctx.register_forward_hook(lambda m, i, o: buf.__setitem__('cw', o[1])),
               model.enc.register_forward_hook(lambda m, i, o: buf.__setitem__('enc', o))]
    K = {k: [] for k in ('P', 'lat', 'enc', 'attr', 'attr_sd', 'cw', 'eff', 'alpha', 'temp',
                         'gate', 'A_test')}
    A_sum = np.zeros((int(Sb.shape[1]),) * 2); C_sum = np.zeros_like(A_sum); n_te = 0
    row = 0
    with torch.no_grad():
        for b in make_loader_s(origins, False, xn, xg):
            xnb, xgb = b[0].to(DEVICE), b[1].to(DEVICE)
            out = model(xnb, xgb, Sb, Sx, Ax)
            B = xnb.shape[0]
            te = torch.as_tensor(np.arange(row, row + B) >= n_tv, device=xnb.device)
            row += B
            Gc, At, aux = buf['cmil']
            if te.any():
                K['P'].append(out[0][te].float().cpu().numpy())
                A_sum += At[te].double().sum(0).cpu().numpy()
                C_sum += aux['Actx'][te].double().sum(0).cpu().numpy()
                n_te += int(te.sum())
                if nm in PER_ORIGIN_A:
                    K['A_test'].append(At[te].half().cpu().numpy())
            if full:
                g = model.gate_min + model.gate_span * torch.sigmoid(buf['g'])
                K['lat'].append(Gc.mean(1).float().cpu().numpy())
                K['enc'].append(buf['enc'].reshape(B, -1, buf['enc'].shape[-1]).mean(1).float().cpu().numpy())
                K['attr'].append(buf['natt'].mean(1).float().cpu().numpy())
                K['attr_sd'].append(buf['natt'].std(1).float().cpu().numpy())
                K['cw'].append(buf['cw'].float().cpu().numpy())
                Pm = At.float().clamp_min(1e-12)
                Pm = Pm / Pm.sum(-1, keepdim=True)
                K['eff'].append(torch.exp(-(Pm * Pm.log()).sum(-1)).mean(-1).cpu().numpy())
                for key, ak in (('alpha', 'alpha'), ('temp', 'temperature')):
                    K[key].append(aux[ak].float().reshape(B, -1).mean(1).cpu().numpy())
                K['gate'].append(g.float().mean((1, 2)).cpu().numpy())
    for h in hs:
        h.remove()
    wm, mae = score_pred(np.concatenate(K['P'], 0))
    assert wm.shape[1] == len(TEST_ORIGINS), wm.shape
    pay = dict(version=np.array(EXT_VERSION), win_mae=wm, mae=mae,
               A_mean=(A_sum / n_te).astype(np.float32), Actx_mean=(C_sum / n_te).astype(np.float32),
               n_params=np.array(sum(p.numel() for p in model.parameters())))
    if full:
        for k in ('lat', 'enc', 'attr', 'attr_sd', 'cw', 'eff', 'alpha', 'temp', 'gate'):
            pay[k] = np.concatenate(K[k], 0).astype(np.float32)
        pay['domains'] = np.array(list(model.ctx.domain_names))
    if K['A_test']:
        pay['A_test'] = np.concatenate(K['A_test'], 0)
    np.savez_compressed(ext_path(nm, seed_), **pay)
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return True


def _fresh(nm, seed_):
    f = ext_path(nm, seed_)
    if not os.path.exists(f):
        return False
    try:
        return str(np.load(f)['version']) == EXT_VERSION
    except Exception:
        return False


jobs = ([(nm, s, True, False) for nm in FULL for s in range(N_SEEDS)]
        + [(nm, s, False, False) for nm in ACC + PROV for s in range(N_SEEDS)]
        + [(UNTRAINED, s, False, True) for s in UNTRAINED_SEEDS])
t0, n_new, n_skip = time.time(), 0, 0
for nm, seed_, full, unt in jobs:
    if _fresh(nm, seed_):
        continue
    if not unt and not os.path.exists(rung_ckpt(nm, seed_)):
        n_skip += 1
        continue
    if extract(nm, seed_, full, unt):
        n_new += 1
print('extracted %d model-seeds in %.1f min; %d had no checkpoint' % (n_new, (time.time() - t0) / 60, n_skip))
for nm in FULL + ACC + PROV + [UNTRAINED]:
    print('  %-24s %2d seeds extracted' % (nm, len(ext_seeds(nm))))

In [ ]:
# === F3. Do the checkpoints reproduce the paper's published tables? ===
# Seed 0 of several rungs was trained in the original session; seeds 1-9 were retrained in
# a later one, on whatever GPU and cuDNN Colab assigned.  Same code, same seeds -- but if the
# environment moved the numbers, the paper must be regenerated from these checkpoints rather
# than quote the old store.  This cell says which.
rows = []
for nm, pub in PUBLISHED.items():
    for seed_ in ext_seeds(nm):
        mae = load_ext(nm, seed_)['mae']
        for i, h in enumerate(HZ):
            rows.append(dict(rung=nm, seed=seed_, h=h, mae=float(mae[i]), published=pub[i]))
REPRO = pd.DataFrame(rows)
REPRO.to_csv(os.path.join(FIN_DIR, 'f3_reproduction_by_seed.csv'), index=False)
if len(REPRO):
    g = REPRO.groupby(['rung', 'h']).agg(n=('seed', 'nunique'), mae=('mae', 'mean'),
                                         published=('published', 'first')).reset_index()
    g['pct'] = 100 * (g.mae - g.published) / g.published
    g.to_csv(os.path.join(FIN_DIR, 'f3_reproduction.csv'), index=False)
    print('=== reproduction: ten-seed mean from these checkpoints vs the published table ===')
    print(g.pivot(index='rung', columns='h', values='pct').round(2).to_string())
    worst = float(g.pct.abs().max())
    print('\nworst deviation %.2f%%.  Within 2%% the published tables stand; beyond it, the tables'
          '\nfor that rung are regenerated from these checkpoints.' % worst)

In [ ]:
# === F4. Experiment 1 at ten seeds: does the pooled latent carry the province-wide factor? ===
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from scipy import stats

city_lp = (df.groupby(['tanggal', 'kab_kota'])['harga']
             .apply(lambda s: np.log(s.dropna()).mean())
             .unstack('kab_kota').sort_index().reindex(dates).ffill(limit=7))
FACT, FMETA = {}, []
for h in HZ:
    Rr = city_lp.shift(-h) - city_lp
    good = Rr.notna().all(axis=1)
    istr = (Rr.index < TEST_START) & good
    mu_, sd_ = Rr[istr].mean(), Rr[istr].std(ddof=0).replace(0, np.nan)
    Z = (Rr - mu_) / sd_
    w, V = np.linalg.eigh(np.cov(Z[istr].to_numpy(), rowvar=False))
    load = V[:, -1] if V[:, -1].sum() > 0 else -V[:, -1]
    F_ = pd.Series(Z.to_numpy() @ load, index=Z.index).where(good)
    FACT[h] = (F_ / F_[istr].std(ddof=0)).reindex(ORIG_DATE).to_numpy()
    Zte = Z[(Z.index >= TEST_START) & good].to_numpy()
    FMETA.append(dict(h=h, pc1_share_train=float(w[-1] / w.sum()),
                      pc1_share_test_frozen=float(np.var(Zte @ load) / np.trace(np.cov(Zte, rowvar=False)))))
pd.DataFrame(FMETA).to_csv(os.path.join(FIN_DIR, 'f4_factor_target.csv'), index=False)

res = []
for nm, desc in EXP1_ARMS:
    for seed_ in ext_seeds(nm):
        e = load_ext(nm, seed_)
        for which in ('lat', 'enc'):
            X = e[which]
            for h in HZ:
                y = FACT[h]
                m = np.isfinite(y)
                tr_, te_ = m & ~IS_TEST, m & IS_TEST
                sc = StandardScaler().fit(X[tr_])
                r_ = RidgeCV(alphas=np.logspace(-2, 5, 30)).fit(sc.transform(X[tr_]), y[tr_])
                p = r_.predict(sc.transform(X[te_]))
                yt = y[te_]
                res.append(dict(rung=nm, arm=desc, latent=which, seed=seed_, h=h,
                                r=float(np.corrcoef(p, yt)[0, 1]),
                                r2=float(1 - ((yt - p) ** 2).sum() / ((yt - yt.mean()) ** 2).sum())))
E1 = pd.DataFrame(res)
E1.to_csv(os.path.join(FIN_DIR, 'f4_factor_probe_by_seed.csv'), index=False)

S1 = (E1.groupby(['latent', 'rung', 'arm', 'h'])
        .agg(n=('seed', 'nunique'), r=('r', 'mean'), r_sd=('r', 'std'), r2=('r2', 'mean'))
        .reset_index())
S1['r_ci95'] = [ci95(E1[(E1.latent == a.latent) & (E1.rung == a.rung) & (E1.h == a.h)].r)
                for a in S1.itertuples()]
S1.to_csv(os.path.join(FIN_DIR, 'f4_factor_probe.csv'), index=False)
print('=== Experiment 1 at ten seeds (test year, ridge fitted on train) ===')
print(S1[S1.latent == 'lat'].pivot(index='arm', columns='h', values='r').round(3).to_string())
print('\nR^2:')
print(S1[S1.latent == 'lat'].pivot(index='arm', columns='h', values='r2').round(3).to_string())

# Arm vs the no-graph arm: independent training runs, so Welch.  Latent vs encoder within one
# model and seed: paired.
cmp_ = []
base = E1[(E1.latent == 'lat') & (E1.rung == 'L1_encoder_only@R')]
for nm, desc in EXP1_ARMS[1:]:
    for h in HZ:
        a = E1[(E1.latent == 'lat') & (E1.rung == nm) & (E1.h == h)].r
        b = base[base.h == h].r
        if len(a) > 1 and len(b) > 1:
            cmp_.append(dict(kind='arm_vs_no_graph', rung=nm, h=h, diff=a.mean() - b.mean(),
                             p=float(stats.ttest_ind(a, b, equal_var=False).pvalue)))
    for h in HZ:
        L = E1[(E1.latent == 'lat') & (E1.rung == nm) & (E1.h == h)].set_index('seed').r
        En = E1[(E1.latent == 'enc') & (E1.rung == nm) & (E1.h == h)].set_index('seed').r
        j = L.index.intersection(En.index)
        if len(j) > 1:
            cmp_.append(dict(kind='pooled_vs_encoder', rung=nm, h=h, diff=float((L[j] - En[j]).mean()),
                             p=float(stats.ttest_rel(L[j], En[j]).pvalue)))
CMP1 = pd.DataFrame(cmp_)
CMP1.to_csv(os.path.join(FIN_DIR, 'f4_factor_probe_contrasts.csv'), index=False)
print('\ncontrasts (diff in r):')
print(CMP1.round(4).to_string(index=False))

In [ ]:
# === F5. Regime tables at ten seeds, both heads, paired statistics ===
# The effect is a PAIRED difference -- the same seed in two regimes -- so its uncertainty is
# the spread of those differences, tested with a paired interval and a sign test.  Density
# is effective degree, exp(row entropy), the threshold-free definition H6 used.
lab_all = regime_labels(ALL_ORIGINS)
rows = []
for nm in EXP2_RUNGS:
    for seed_ in ext_seeds(nm):
        e = load_ext(nm, seed_)
        doms = [str(x) for x in e['domains']]
        meas = {('attr_' + d): e['attr'][:, j] for j, d in enumerate(doms)}
        meas.update(eff_degree=e['eff'], alpha=e['alpha'], temperature=e['temp'], gate=e['gate'])
        for mname, v in meas.items():
            for g_, rg in REG.items():
                sel = lab_all == g_
                if sel.sum():
                    rows.append(dict(rung=nm, seed=seed_, measure=mname, regime=rg, value=float(v[sel].mean())))
PER = pd.DataFrame(rows)
PER.to_csv(os.path.join(FIN_DIR, 'f5_regime_by_seed.csv'), index=False)

out = []
for (nm, mname), grp in PER.groupby(['rung', 'measure']):
    wide = grp.pivot(index='seed', columns='regime', values='value')
    for rg in ('ramadan', 'peak_rain', 'harvest'):
        if rg not in wide or 'normal' not in wide:
            continue
        dv = (wide[rg] - wide['normal']).dropna().to_numpy()
        lvl = wide['normal'].mean()
        out.append(dict(rung=nm, measure=mname, regime=rg, n=len(dv), normal=lvl,
                        normal_ci95=ci95(wide['normal']), effect=dv.mean(), effect_ci95=ci95(dv),
                        rel_pct=100 * dv.mean() / lvl if lvl else np.nan,
                        agree=int(max((dv > 0).sum(), (dv < 0).sum())), sign_p=sign_test(dv)))
REG10 = pd.DataFrame(out)
REG10['survives'] = (REG10.sign_p < 0.05) & (REG10.effect.abs() > REG10.effect_ci95)
REG10.to_csv(os.path.join(FIN_DIR, 'f5_regime_effects.csv'), index=False)
for nm in EXP2_RUNGS:
    s = REG10[REG10.rung == nm]
    print('=== %s: %d of %d regime effects survive the paired interval and the sign test ==='
          % (nm, int(s.survives.sum()), len(s)))
    print(s[s.survives][['measure', 'regime', 'normal', 'effect', 'effect_ci95', 'rel_pct',
                          'agree', 'sign_p']].round(4).to_string(index=False))
    att = s[s.measure.str.startswith('attr_')]
    print('  attribution effects surviving: %d of %d\n' % (int(att.survives.sum()), len(att)))

In [ ]:
# === F6. Event alignment at ten seeds, both heads, full sensitivity grid ===
# Pre-specified exactly as in the validation notebook: trailing-60-day 90th-percentile peak
# rule, +/-3-day match, per-type circular-shift null, domain-specificity control.
W_TRAIL, Q_TRAIL, MATCH_DAYS, N_NULL, SEED0 = 60, 0.90, 3, 1000, 20260909

ev = []
prec = df.groupby('tanggal')['nasa_PRECTOTCORR'].mean().reindex(dates)
thr_rain = prec[prec.index < TEST_START].quantile(0.95)
for d0 in prec[prec >= thr_rain].index:
    ev.append((d0, 'weather', 'extreme_rain'))
FUEL = ['Harga BBM (Pertalite)', 'Harga BBM (Pertamax)', 'Harga BBM (Solar Subsidi)']
gg = df[['tanggal', 'Ramadan'] + FUEL].drop_duplicates('tanggal').set_index('tanggal').sort_index().reindex(dates)
rr = gg.Ramadan.fillna(0).astype(int)
for _, blk in gg[rr == 1].groupby((rr.diff() != 0).cumsum()[rr == 1]):
    ev += [(d0, 'economy', 'ramadan') for d0 in blk.index]
    nxt = blk.index.max() + pd.Timedelta(days=1)
    if nxt in dates:
        ev.append((nxt, 'economy', 'idul_fitri'))
ev += [(d0, 'economy', 'fuel_price_change') for d0 in gg.index[gg[FUEL].diff().abs().sum(axis=1) > 0]]
SUPC = ['market_operation', 'import_export', 'supply_reserve', 'procurement_distribution',
        'production', 'price_management', 'subsidy_assistance']
pol = policy_events_all[(policy_events_all.event_date >= dates.min()) & (policy_events_all.event_date <= dates.max())]
ev += [(x.event_date, 'supply', 'policy') for _, x in pol[pol[SUPC].sum(axis=1) > 0].iterrows()]
EVL = pd.DataFrame(ev, columns=['date', 'domain', 'event_type'])
EVL = EVL[EVL.date.isin(dates)].drop_duplicates(['date', 'event_type']).sort_values('date')
EVL.to_csv(os.path.join(FIN_DIR, 'f6_event_log.csv'), index=False)
print('events:', EVL.groupby(['domain', 'event_type']).size().to_dict())


def _dilate(idx, n, k):
    b = np.zeros(n, np.int8)
    b[idx] = 1
    return (np.convolve(b, np.ones(2 * k + 1, np.int8), 'same') > 0) if k else b.astype(bool)


def align_one(A, q=None, k=None, with_raw=True):
    q = Q_TRAIL if q is None else q
    k = MATCH_DAYS if k is None else k
    A = A.sort_index()
    idx = A.index; n = len(idx); pos = {d0: i for i, d0 in enumerate(idx)}
    e = EVL[EVL.date.isin(idx)].copy(); e['i'] = e.date.map(pos).astype(int)
    thr = A.shift(1).rolling(W_TRAIL, min_periods=W_TRAIL // 2).quantile(q)
    P = (A > thr) & thr.notna()
    pk = {d: np.flatnonzero(P[d].to_numpy()) for d in A.columns}
    dpk = {d: _dilate(pk[d], n, k) for d in A.columns}
    types = {d: [g.i.to_numpy() for _, g in e[e.domain == d].groupby('event_type')] for d in A.columns}
    rng = np.random.default_rng(SEED0)
    rec = lambda d, ei: float(dpk[d][ei].mean()) if len(ei) else np.nan
    draw = lambda d: (np.concatenate([(t + rng.integers(0, n)) % n for t in types[d]])
                      if types[d] else np.array([], int))
    raw, spec, conf = [], [], []
    for dom in A.columns:
        ei = e.loc[e.domain == dom, 'i'].to_numpy()
        if with_raw:
            for de in A.columns:
                ee = e.loc[e.domain == de, 'i'].to_numpy()
                conf.append(dict(peak_domain=dom, event_domain=de,
                                 share=float(_dilate(ee, n, k)[pk[dom]].mean()) if len(ee) and len(pk[dom]) else np.nan))
            if len(ei) and len(pk[dom]):
                rc = rec(dom, ei); pr = float(_dilate(ei, n, k)[pk[dom]].mean())
                f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
                nul = []
                for _ in range(N_NULL):
                    si = draw(dom); r2 = rec(dom, si); p2 = float(_dilate(si, n, k)[pk[dom]].mean())
                    nul.append(2 * p2 * r2 / (p2 + r2) if p2 + r2 else 0.0)
                raw.append(dict(domain=dom, n_events=len(ei), precision=pr, recall=rc, f1=f1,
                                null_f1=float(np.mean(nul)), p=float((np.array(nul) >= f1).mean())))
        if len(ei):
            oth = [d for d in A.columns if d != dom]
            own, others = rec(dom, ei), float(np.nanmean([rec(d, ei) for d in oth]))
            nul = np.empty(N_NULL)
            for b in range(N_NULL):
                si = draw(dom)
                nul[b] = rec(dom, si) - np.nanmean([rec(d, si) for d in oth])
            spec.append(dict(event_domain=dom, n_events=len(ei), own=own, others=others,
                             specificity=own - others, p=float((nul >= own - others).mean())))
    return pd.DataFrame(raw), pd.DataFrame(spec), pd.DataFrame(conf)


RAW, SPEC, CONF, DOF = [], [], [], []
for nm in EXP2_RUNGS:
    for seed_ in ext_seeds(nm):
        e = load_ext(nm, seed_)
        doms = [str(x) for x in e['domains']]
        for series, arr in (('node', e['attr']), ('global', e['cw'])):
            A = pd.DataFrame(arr, index=ORIG_DATE, columns=doms)
            X = (A - A.mean()) / A.std()
            ev_ = np.linalg.eigvalsh(np.cov(X.dropna().to_numpy(), rowvar=False))[::-1]
            DOF.append(dict(rung=nm, series=series, seed=seed_, pc1_share=float(ev_[0] / ev_.sum())))
            r_, s_, c_ = align_one(A, with_raw=(series == 'node'))
            for t_, L_ in ((r_, RAW), (s_, SPEC), (c_, CONF)):
                if len(t_):
                    t_['rung'], t_['series'], t_['seed'] = nm, series, seed_
                    L_.append(t_)
RAW, SPEC, CONF, DOF = (pd.concat(RAW), pd.concat(SPEC), pd.concat(CONF), pd.DataFrame(DOF))
for t_, f_ in ((RAW, 'f6_alignment_raw'), (SPEC, 'f6_specificity'), (CONF, 'f6_confusion'), (DOF, 'f6_dof')):
    t_.to_csv(os.path.join(FIN_DIR, f_ + '.csv'), index=False)

print('\n=== degrees of freedom (PC1 share of the four weights; 1.0 = one scalar) ===')
print(DOF.groupby(['rung', 'series']).pc1_share.agg(['mean', 'min', 'max']).round(3).to_string())
print('\n=== domain specificity, node-level attribution ===')
print(SPEC[SPEC.series == 'node'].groupby(['rung', 'event_domain']).agg(
    n=('n_events', 'first'), specificity=('specificity', 'mean'), p_median=('p', 'median'),
    seeds_p05=('p', lambda s: int((s < 0.05).sum()))).round(3).to_string())

sens = []
nmS = EXP2_RUNGS[0]
for seed_ in ext_seeds(nmS):
    e = load_ext(nmS, seed_)
    A = pd.DataFrame(e['attr'], index=ORIG_DATE, columns=[str(x) for x in e['domains']])
    for q in (0.85, 0.90, 0.95):
        for k in (0, 1, 3, 7):
            s_ = align_one(A, q=q, k=k, with_raw=False)[1]
            s_['q'], s_['k'], s_['seed'] = q, k, seed_
            sens.append(s_)
SENS = pd.concat(sens)
SENS.to_csv(os.path.join(FIN_DIR, 'f6_sensitivity.csv'), index=False)
print('\n=== sensitivity (%s, all seeds): max specificity and min median p over the 12 settings ===' % nmS)
agg = SENS.groupby(['event_domain', 'q', 'k']).agg(spec=('specificity', 'median'), p=('p', 'median'))
print(agg.groupby('event_domain').agg(max_spec=('spec', 'max'), min_p=('p', 'min')).round(3).to_string())

In [ ]:
# === F7. Adjacency reproducibility: learned structure, or the shared anchor? ===
# A_t = alpha * A_phys + (1 - alpha) * A_ctx.  Every seed shares A_phys, so the full adjacency
# correlates across seeds before a single step of training.  The paper's r = 0.987 was
# computed on the full adjacency with no floor.  Here it is computed three ways:
#   on A_t, on the learned part A_ctx, and for ten UNTRAINED models as the floor.
# Only reproducibility of A_ctx above its untrained floor says the learned graph is a
# property of the data.
off = ~np.eye(R, dtype=bool)


def edge_r(mats, offdiag=True):
    if len(mats) < 2:
        return dict(n=len(mats), r=np.nan, r_min=np.nan)
    sel = (lambda M: M[off]) if offdiag else (lambda M: M.ravel())
    rs = [float(np.corrcoef(sel(a), sel(b))[0, 1]) for a, b in itertools.combinations(mats, 2)]
    return dict(n=len(mats), pairs=len(rs), r=float(np.mean(rs)), r_sd=float(np.std(rs)), r_min=float(np.min(rs)))


rows = []
for nm in ['L6_ctx_conditioned@S', 'L6_ctx_conditioned@R', 'L7_ctx_shuffled@S', 'H_real@S', UNTRAINED]:
    ss = ext_seeds(nm)
    if len(ss) < 2:
        print('  %-22s %d seeds extracted -- skipped' % (nm, len(ss)))
        continue
    E = [load_ext(nm, s) for s in ss]
    for part in ('A_mean', 'Actx_mean'):
        for od in (False, True):
            rows.append(dict(rung=nm, part=part, offdiag=od, **edge_r([e[part] for e in E], od)))
REP = pd.DataFrame(rows)
REP.to_csv(os.path.join(FIN_DIR, 'f7_reproducibility.csv'), index=False)
print('=== edge correlation across seed pairs ===')
print(REP.pivot_table(index='rung', columns=['part', 'offdiag'], values='r').round(3).to_string())
print('\nread A_mean with offdiag=False against the paper\'s 0.987; read Actx_mean against the')
print('UNTRAINED row.  If the trained A_ctx is not clearly above its floor, the reproducibility')
print('claim is a property of the anchor and must be restated as such.')

In [ ]:
# === F8. Regime response of the adjacency, and its shifted-context placebo, at ten seeds ===
def _f_ratio(Aflat, lab):
    gm = Aflat.mean(0); between = within = 0.0
    for g_ in np.unique(lab):
        m = lab == g_
        if m.sum() < 2:
            continue
        between += m.sum() * ((Aflat[m].mean(0) - gm) ** 2).sum()
        within += ((Aflat[m] - Aflat[m].mean(0)) ** 2).sum()
    return float(between / max(within, 1e-12))


lab_te = regime_labels(TEST_ORIGINS)
rows = []
for nm in ['L6_ctx_conditioned@S', 'L6_ctx_conditioned@R', 'L7_ctx_shuffled@S']:
    for seed_ in ext_seeds(nm):
        e = load_ext(nm, seed_)
        if 'A_test' not in e:
            continue
        Af = e['A_test'].astype(np.float32).reshape(len(e['A_test']), -1)
        obs_F = _f_ratio(Af, lab_te)
        rng = np.random.default_rng(7)
        null = np.array([_f_ratio(Af, np.roll(lab_te, int(o))) for o in rng.integers(1, len(lab_te), 1000)])
        rows.append(dict(rung=nm, seed=seed_, F=obs_F, null_median=float(np.median(null)),
                         F_over_null=obs_F / float(np.median(null)), p=float((null >= obs_F).mean())))
RR = pd.DataFrame(rows)
RR.to_csv(os.path.join(FIN_DIR, 'f8_regime_response_by_seed.csv'), index=False)
if len(RR):
    print('=== regime response of A_t (circular-shift null on the regime labels) ===')
    print(RR.groupby('rung').agg(n=('seed', 'nunique'), F_over_null=('F_over_null', 'mean'),
                                 p_median=('p', 'median'), seeds_p05=('p', lambda s: int((s < 0.05).sum())))
          .round(3).to_string())
    a = RR[RR.rung == 'L6_ctx_conditioned@S'].F_over_null
    b = RR[RR.rung == 'L7_ctx_shuffled@S'].F_over_null
    if len(a) > 1 and len(b) > 1:
        from scipy import stats
        print('\nreal context vs shifted context, F/null: %.3f vs %.3f, Mann-Whitney p = %.4f'
              % (a.mean(), b.mean(), stats.mannwhitneyu(a, b, alternative='greater').pvalue))

In [ ]:
# === F9. Market-level tests at ten seeds: co-movement, and density vs dispersion ===
_scale_intact()
P_idr = np.exp(Y * sd[None, :] + mu[None, :])
CITY_P = np.full((P_idr.shape[0], R), np.nan, np.float32)
for a in range(R):
    ix = np.where(S[:, a] > 0)[0]
    w = (obs[:, ix] > 0).astype(np.float32); den = w.sum(1)
    CITY_P[:, a] = np.where(den > 0, (P_idr[:, ix] * w).sum(1) / np.maximum(den, 1.0), np.nan)
CITY_P = pd.DataFrame(CITY_P).ffill().bfill().to_numpy(np.float32)
DISPERSION = CITY_P.std(1) / np.maximum(CITY_P.mean(1), 1e-9)
ret = np.diff(np.log(np.maximum(CITY_P, 1e-9)), axis=0)[TEST_ORIGINS[0]:TEST_ORIGINS[-1]]
Cm = np.nan_to_num(np.corrcoef(np.nan_to_num(ret).T))
sym = lambda M: 0.5 * (M + M.T)
r_phys = float(np.corrcoef(sym(A_phys)[off], Cm[off])[0, 1])

lab_n_te = len(TEST_ORIGINS)
rows = []
for nm in ['L6_ctx_conditioned@S', 'L6_ctx_conditioned@R']:
    for seed_ in ext_seeds(nm):
        e = load_ext(nm, seed_)
        row = dict(rung=nm, seed=seed_, r_phys=r_phys,
                   r_learned=float(np.corrcoef(sym(e['A_mean'])[off], Cm[off])[0, 1]),
                   r_ctx=float(np.corrcoef(sym(e['Actx_mean'])[off], Cm[off])[0, 1]))
        # density vs dispersion over the test origins; the null is a circular shift of the
        # dispersion series, and the statistic is the MAXIMUM |r| over lags, so choosing the
        # best lag cannot manufacture significance.
        dens = e['eff'][IS_TEST]
        disp = DISPERSION[list(TEST_ORIGINS)]
        lags = range(-7, 8)

        def best(y):
            out = []
            for L_ in lags:
                x_ = dens[max(0, -L_):len(dens) - max(0, L_)]
                y_ = y[max(0, L_):len(y) - max(0, -L_)]
                out.append(np.corrcoef(x_, y_)[0, 1])
            j = int(np.nanargmax(np.abs(out)))
            return list(lags)[j], float(out[j])
        lag, r_ = best(disp)
        rng = np.random.default_rng(11)
        nul = np.array([abs(best(np.roll(disp, int(o)))[1]) for o in rng.integers(1, len(disp), 500)])
        row.update(disp_lag=lag, disp_r=r_, disp_p=float((nul >= abs(r_)).mean()))
        rows.append(row)
MKT = pd.DataFrame(rows)
MKT.to_csv(os.path.join(FIN_DIR, 'f9_market_level.csv'), index=False)
if len(MKT):
    print('=== market-level tests, mean over seeds ===')
    print(MKT.groupby('rung').agg(n=('seed', 'nunique'), r_learned=('r_learned', 'mean'), r_ctx=('r_ctx', 'mean'),
                                  r_phys=('r_phys', 'first'), disp_r=('disp_r', 'mean'),
                                  disp_p_median=('disp_p', 'median')).round(3).to_string())

In [ ]:
# === F10. Is the per-market attribution market-specific, or one attribution copied 120 times? ===
rows = []
for nm in EXP2_RUNGS:
    for seed_ in ext_seeds(nm):
        e = load_ext(nm, seed_)
        for j, d in enumerate([str(x) for x in e['domains']]):
            rows.append(dict(rung=nm, seed=seed_, domain=d, mean_weight=float(e['attr'][:, j].mean()),
                             sd_across_markets=float(e['attr_sd'][:, j].mean()),
                             cv_pct=float(100 * (e['attr_sd'][:, j] / np.maximum(e['attr'][:, j], 1e-9)).mean())))
HOM = pd.DataFrame(rows)
HOM.to_csv(os.path.join(FIN_DIR, 'f10_attribution_homogeneity.csv'), index=False)
if len(HOM):
    print(HOM.groupby(['rung', 'domain']).agg(n=('seed', 'nunique'), mean_weight=('mean_weight', 'mean'),
                                             cv_pct=('cv_pct', 'mean'), cv_max=('cv_pct', 'max')).round(3).to_string())

In [ ]:
# === F11. Mechanism contrasts at ten seeds, with the paper's own inference ===
# Two-level paired bootstrap over seeds and windows, TOST at a 2% margin declared in advance,
# Benjamini-Hochberg within each family.  Identical to H4, applied to the extracted windows.
BOOT_B, MARGIN = 1000, 2.0


_WCACHE = {}


def windows(nm, hi):
    if nm not in _WCACHE:
        _WCACHE[nm] = np.stack([load_ext(nm, s)['win_mae'] for s in ext_seeds(nm)])
    return _WCACHE[nm][:, hi]


def paired_boot(nm_a, nm_b, hi, rng):
    A_, B_ = windows(nm_a, hi), windows(nm_b, hi)
    T_ = A_.shape[1]
    obs_ = A_.mean() - B_.mean()
    dr = np.empty(BOOT_B)
    for i in range(BOOT_B):
        wi = rng.integers(0, T_, T_)
        dr[i] = A_[np.ix_(rng.integers(0, len(A_), len(A_)), wi)].mean() - \
                B_[np.ix_(rng.integers(0, len(B_), len(B_)), wi)].mean()
    lo, hi_ = np.percentile(dr, [2.5, 97.5])
    return obs_, lo, hi_, float(min(1.0, 2 * min((dr <= 0).mean(), (dr >= 0).mean()))), B_.mean(), len(A_), len(B_)


def bh(p, q=0.05):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    ok = p[o] <= q * np.arange(1, n + 1) / n
    k = np.max(np.where(ok)[0]) + 1 if ok.any() else 0
    rej = np.zeros(n, bool); rej[o[:k]] = True
    return rej


REF = 'L6same@R' if len(ext_seeds('L6same@R')) >= 2 else 'L6_ctx_conditioned@R'
print('reference for the mechanism contrasts:', REF)
# Each family is a list of (variant, reference) pairs.  The session check compares L6 retrained
# here against the ORIGINAL L6; if that is equivalent, the session does not matter and every
# other contrast is clean.  If it is not, the mechanism contrasts are still valid because they
# use the same-session replicate as their reference, and the paper says so.
FAMILIES = {'session_check': [('L6same@R', 'L6_ctx_conditioned@R')],
            'one_point_removed': [('L6no%s@R' % k, REF) for k in PATCHES],
            'head_count': [(v, REF) for v in ('L6nh1@R', 'L6nh2@R', 'L6nh8@R')],
            'width': [(v, REF) for v in ('L6d32@R', 'L6d128@R')]}
rng = np.random.default_rng(0)
allc = []
for fam, pairs in FAMILIES.items():
    rows = []
    for nm, ref_nm in pairs:
        if nm == ref_nm or len(ext_seeds(nm)) < 2 or len(ext_seeds(ref_nm)) < 2:
            continue
        for hi, h in enumerate(HZ):
            d_, lo, hi_, p, ref, na, nb = paired_boot(nm, ref_nm, hi, rng)
            m = MARGIN / 100 * ref
            inside, excl = (lo > -m) and (hi_ < m), (lo > 0) or (hi_ < 0)
            rows.append(dict(family=fam, variant=nm, reference=ref_nm, h=h, n_variant=na, n_ref=nb,
                             diff=d_, lo=lo, hi=hi_, pct=100 * d_ / ref, p=p,
                             verdict='equivalent' if inside else ('different' if excl else 'inconclusive')))
    if rows:
        D = pd.DataFrame(rows)
        D['bh_reject'] = bh(D.p.values)
        D.loc[(D.verdict == 'different') & ~D.bh_reject, 'verdict'] = 'inconclusive'
        allc.append(D)
        print('=== %s (positive pct = worse than the reference) ===' % fam)
        print(D.pivot(index='variant', columns='h', values='pct').round(2).to_string())
        print(D.groupby('variant').verdict.value_counts().unstack(fill_value=0).to_string(), '\n')
CON = pd.concat(allc) if allc else pd.DataFrame()
CON.to_csv(os.path.join(FIN_DIR, 'f11_contrasts.csv'), index=False)
pc = {nm: int(load_ext(nm, ext_seeds(nm)[0])['n_params']) for nm in [REF] + ACC if ext_seeds(nm)}
pd.Series(pc, name='n_params').to_csv(os.path.join(FIN_DIR, 'f11_param_counts.csv'))
print('parameter counts:', pc)

In [ ]:
# === F12. Summary, and one zip to send back ===
print('=' * 78)
print('CMIN FINAL RUN -- SUMMARY')
print('=' * 78)
for nm in FULL + ACC + PROV + [UNTRAINED]:
    print('  %-24s %2d seeds' % (nm, len(ext_seeds(nm))))
if 'REPRO' in globals() and len(REPRO):
    gg_ = REPRO.groupby(['rung', 'h']).agg(mae=('mae', 'mean'), pub=('published', 'first'))
    print('\nF3 worst deviation from the published tables: %.2f%%'
          % float((100 * (gg_.mae - gg_.pub) / gg_.pub).abs().max()))
if 'S1' in globals():
    print('\nF4 factor probe, pooled latent, r (mean over seeds):')
    print(S1[S1.latent == 'lat'].pivot(index='arm', columns='h', values='r').round(3).to_string())
if 'REG10' in globals():
    print('\nF5 surviving regime effects:', REG10.groupby('rung').survives.sum().to_dict())
if 'SPEC' in globals():
    print('\nF6 node specificity, seeds at p<0.05:')
    print(SPEC[SPEC.series == 'node'].groupby(['rung', 'event_domain']).p.apply(lambda s: int((s < 0.05).sum())).to_string())
if 'REP' in globals() and len(REP):
    print('\nF7 reproducibility:')
    print(REP[REP.offdiag].pivot_table(index='rung', columns='part', values='r').round(3).to_string())
if 'RR' in globals() and len(RR):
    print('\nF8 regime response, median p:', RR.groupby('rung').p.median().round(4).to_dict())
if 'CON' in globals() and len(CON):
    print('\nF11 verdicts:')
    print(CON.groupby(['family', 'variant']).verdict.value_counts().unstack(fill_value=0).to_string())

ZIP = os.path.join(FIN_DIR, 'cmin_final_results.zip')
adj = {}
for nm in ['L6_ctx_conditioned@S', 'L6_ctx_conditioned@R', 'L7_ctx_shuffled@S', 'H_real@S', UNTRAINED]:
    for s in ext_seeds(nm):
        e = load_ext(nm, s)
        adj['%s|%d|A' % (nm, s)] = e['A_mean']; adj['%s|%d|Actx' % (nm, s)] = e['Actx_mean']
np.savez_compressed(os.path.join(FIN_DIR, 'adjacency_means.npz'), **adj)
daily = []
for nm in FULL:
    for s in ext_seeds(nm):
        e = load_ext(nm, s)
        d = pd.DataFrame({'rung': nm, 'seed': s, 'origin_date': ORIG_DATE, 'is_test': IS_TEST,
                          'eff_degree': e['eff'], 'alpha': e['alpha'], 'temperature': e['temp'], 'gate': e['gate']})
        for j, dn in enumerate([str(x) for x in e['domains']]):
            d['attr_' + dn] = e['attr'][:, j]; d['ctxw_' + dn] = e['cw'][:, j]
        daily.append(d)
if daily:
    pd.concat(daily).to_csv(os.path.join(FIN_DIR, 'daily_series.csv.gz'), index=False, compression='gzip')
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(FIN_DIR)):
        p = os.path.join(FIN_DIR, f)
        if os.path.isfile(p) and f != os.path.basename(ZIP):
            z.write(p, f)
print('\nwrote %s (%.1f MB).  Download it and send it back.' % (ZIP, os.path.getsize(ZIP) / 1e6))

## What each result does to the paper

| cell | if it comes back... | the paper |
|---|---|---|
| F3 | within 2% of the published tables | the tables stand as printed |
| F3 | beyond 2% for some rung | that rung's tables are regenerated from these checkpoints |
| F4 | pooled latent tracks the factor at ten seeds | Experiment 1 is claimed with confidence intervals, ordering included if resolved |
| F5, F6 | as at ten seeds before | the withdrawal of the attribution claim is confirmed on both heads |
| F7 | learned part reproducible above its untrained floor | the reproducibility claim stands, restated on the learned part |
| F7 | only the full adjacency is reproducible | the claim is restated as a property of the anchor, and r = 0.987 is removed |
| F8 | real context beats the placebo across seeds | "adjacency responds to context" is claimed at ten seeds |
| F11 | a one-point variant is equivalent to L6 | that conditioning point is reported as inert, each on its own evidence |
| F11 | a one-point variant is worse | that point is the one doing the work, and Section 6.8 says so |